# MICrONS v13: the self-critique, fixed

Built from commit `5420e9f`. The questions, primary metrics and tests were fixed in `PREREGISTRATION.md`
(committed before this run); hyperparameters are tuned inside this session on the validation split, and every reported number is on
the test split, which nothing is tuned on. 5 seeds, paired comparisons, Holm-Bonferroni across 7 primary tests.

1. **Baselines** - the single brain against a tuned, parameter-matched GRU and character transformer.
2. **Learning on the job** - a gated delta-rule fast memory trained end to end; does carrying it across a stream
   help with no gradient at all? Plus the readout delta rule, and dynamic evaluation as the backprop ceiling.
3. **Differentiation** - six brains that each grow their own synapses where their own loss gradient points (RigL),
   against six identical brains, with specialisation measured on the frozen model.
4. **Neural prediction** - does the sparse E/I brain still beat a GRU once the GRU is tuned (v6's claim)?

Not tested, and why: whether the *measured* wiring matters needs MICrONS functional coregistration (via CAVE,
unavailable), so v13 makes no claim about the connectome itself.

In [ ]:
import time
T0 = time.time()  # the 12 h clock starts with the session
SEEDS = 5
DEADLINE = T0 + 11.25 * 3600
COMMIT = "5420e9f"

In [ ]:
import os, torch
for d in ("/kaggle/working/multiverse/data", "/kaggle/working/multiverse/tests", "/kaggle/working/multiverse/results", "/kaggle/working/char_lm"):
    os.makedirs(d, exist_ok=True)
print("torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
%%writefile /kaggle/working/multiverse/connectome.py
from dataclasses import dataclass, replace

import h5py
import numpy as np
import pandas as pd

MICRONS_H5 = r"E:\microns_data\microns_mm3_connectome_v1181.h5"


@dataclass
class Connectome:
    name: str
    cell_type: np.ndarray
    inhibitory: np.ndarray
    pre: np.ndarray
    post: np.ndarray
    weight: np.ndarray
    input_types: tuple
    output_types: tuple
    xyz: np.ndarray | None = None

    @property
    def n(self):
        return len(self.cell_type)

    @property
    def input_idx(self):
        return np.flatnonzero(np.isin(self.cell_type, self.input_types))

    @property
    def output_idx(self):
        return np.flatnonzero(np.isin(self.cell_type, self.output_types))

    def shuffled(self, seed=0):
        """Degree-preserving null: every neuron keeps its in/out degree, partners are scrambled."""
        perm = np.random.default_rng(seed).permutation(len(self.post))
        return replace(self, name=f"{self.name}-shuffled", post=self.post[perm])

    def randomized(self, seed=0):
        """Erdos-Renyi null: same neuron and synapse counts, partners drawn uniformly (no connectome structure)."""
        rng = np.random.default_rng(seed)
        pre, post = rng.integers(0, self.n, len(self.pre)), rng.integers(0, self.n, len(self.post))
        keep = pre != post
        return replace(self, name=f"{self.name}-random", pre=pre[keep], post=post[keep], weight=self.weight[keep])

    def summary(self):
        types, counts = np.unique(self.cell_type, return_counts=True)
        mix = ", ".join(f"{t}:{c}" for t, c in zip(types, counts))
        return (f"{self.name}: {self.n} neurons, {len(self.pre)} connections, "
                f"{self.inhibitory.mean():.0%} inhibitory, input {len(self.input_idx)} / readout "
                f"{len(self.output_idx)} neurons [{mix}]")


def load_microns(path=MICRONS_H5, proofread="axon"):
    """Proofread neurons from MICrONS minnie65 v1181 and the condensed connections among them."""
    vp = pd.read_hdf(path, "/connectivity/condensed/vertex_properties")
    column = "status_axon" if proofread == "axon" else "status_dendrite"
    keep = (vp[column] == "t").to_numpy()
    with h5py.File(path, "r") as f:
        ij = f["connectivity/condensed/edge_indices/block0_values"][:]
        items = [x.decode() for x in f["connectivity/condensed/edges/block0_items"][:]]
        size = f["connectivity/condensed/edges/block0_values"][:, items.index("total_size")]
    new_id = np.full(len(vp), -1)
    new_id[keep] = np.arange(keep.sum())
    m = keep[ij[:, 0]] & keep[ij[:, 1]] & (ij[:, 0] != ij[:, 1])
    sub = vp[keep]
    return Connectome(
        name=f"microns-v1181-{proofread}",
        cell_type=sub["cell_type"].to_numpy(dtype=str),
        inhibitory=(sub["classification_system"] == "inhibitory_neuron").to_numpy(),
        pre=new_id[ij[m, 0]],
        post=new_id[ij[m, 1]],
        weight=size[m].astype(np.float64),
        input_types=("L4a", "L4b", "L4c"),
        output_types=("L5ET", "L5NP", "L5a", "L5b"),
        xyz=sub[["x_nm", "y_nm", "z_nm"]].to_numpy(dtype=float),
    )


def save_npz(conn, path):
    np.savez_compressed(
        path, name=conn.name, cell_type=conn.cell_type, inhibitory=conn.inhibitory,
        pre=conn.pre.astype(np.int32), post=conn.post.astype(np.int32), weight=conn.weight.astype(np.float32),
        input_types=np.array(conn.input_types), output_types=np.array(conn.output_types),
        xyz=conn.xyz.astype(np.float32),
    )


def load_npz(path):
    z = np.load(path)
    return Connectome(
        name=str(z["name"]), cell_type=z["cell_type"], inhibitory=z["inhibitory"],
        pre=z["pre"].astype(np.int64), post=z["post"].astype(np.int64), weight=z["weight"].astype(np.float64),
        input_types=tuple(z["input_types"].tolist()), output_types=tuple(z["output_types"].tolist()), xyz=z["xyz"],
    )


def synthetic(n=400, k_out=40, seed=0):
    """Stand-in with MICrONS-like statistics: laminar types, ~15% inhibition, distance-dependent, lognormal weights."""
    rng = np.random.default_rng(seed)
    xyz = rng.uniform(0, 1, (n, 3))
    depth = xyz[:, 1]
    excit = np.select([depth < 0.3, depth < 0.45, depth < 0.7], ["L23", "L4", "L5"], "L6")
    inhib = np.where(rng.uniform(size=n) < 0.6, "PTC", "DTC")
    inhibitory = rng.uniform(size=n) < 0.15
    cell_type = np.where(inhibitory, inhib, excit)

    d2 = ((xyz[:, None, :] - xyz[None, :, :]) ** 2).sum(-1)
    prob = np.exp(-d2 / (2 * 0.2 ** 2))
    np.fill_diagonal(prob, 0)
    prob = np.clip(prob / prob.sum(1, keepdims=True) * k_out, 0, 1)
    pre, post = np.nonzero(rng.uniform(size=(n, n)) < prob)
    weight = rng.lognormal(0.0, 1.0, len(pre))
    return Connectome("synthetic", cell_type, inhibitory, pre, post, weight, ("L4",), ("L5",), xyz)

In [ ]:
%%writefile /kaggle/working/multiverse/model.py
import math
import warnings

import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore", message="Sparse CSR tensor support is in beta")

MODES = ("single", "separate", "separate_matched", "classical", "product", "quantum")
REWIRE_RULES = ("none", "random", "like", "local", "like_local", "rigl")
NEW_SYNAPSE_FRAC = 0.1  # grown synapses start at a tenth of typical strength, so growth perturbs instead of shocks
DIAGONAL_MODES = ("separate", "separate_matched")
N_TYPE_PAIRS = 4  # (E,I) x (E,I) for pre/post


def _unitary(re, im):
    A = torch.complex(re, im)
    return torch.linalg.matrix_exp(A - A.conj().transpose(-2, -1))


def _diagonalizing_reflection(K):
    """Householder reflection sending the uniform product state to sum_a |a,a>/sqrt(K)."""
    v = torch.full((K * K,), 1.0 / K)
    t = torch.zeros(K * K)
    t[torch.arange(K) * (K + 1)] = 1.0 / math.sqrt(K)
    w = v - t
    return torch.eye(K * K) - 2 * torch.outer(w, w) / (w @ w)


def _spectral_radius(pre, post, values, n):
    M = sp.csr_matrix((values, (post, pre)), shape=(n, n))
    ev = spla.eigs(M, k=1, which="LM", return_eigenvectors=False, maxiter=10000, tol=1e-3)
    return float(np.abs(ev[0]))


def _faulty(w, faults):
    """Hardware-style faults on effective synapse weights, drawn reproducibly from faults['seed']."""
    if not faults:
        return w
    g = torch.Generator().manual_seed(faults.get("seed", 0))
    if faults.get("weight_noise"):
        w = w * (1 + faults["weight_noise"] * torch.randn(w.shape, generator=g).to(w.device))
    if faults.get("dead_frac"):
        w = w * (torch.rand(w.shape, generator=g) >= faults["dead_frac"]).to(w.device, w.dtype)
    if faults.get("bits"):
        levels = 2 ** faults["bits"] - 1
        top = w.abs().max()
        w = torch.sign(w) * torch.round(w.abs() / top * levels) / levels * top
    return w


def _crow(rows, n):
    crow = torch.zeros(n + 1, dtype=torch.long, device=rows.device)
    crow[1:] = torch.cumsum(torch.bincount(rows, minlength=n), 0)
    return crow


class _Pattern:
    """CSR sparsity pattern of an n x n coupling (duplicates merged) plus its transpose."""

    def __init__(self, dst, src, n):
        uniq, self.inverse = torch.unique(dst * n + src, return_inverse=True)
        self.n = n
        self.rows, self.cols = uniq // n, uniq % n
        self.crow = _crow(self.rows, n)
        self.t_order = torch.argsort(self.cols * n + self.rows)
        self.t_crow = _crow(self.cols, n)
        self.t_cols = self.rows[self.t_order]


class _MeshMul(torch.autograd.Function):
    """Recurrent input J x. vals[e, s] links brain a[e, s] of pre[e] to brain b[e, s] of post[e].
    Forward and dL/dx run as CSR multiplies (torch's sparse COO multiply is ~40x slower on CPU);
    dL/dvals is one brain-by-brain outer product per synapse."""

    @staticmethod
    def forward(ctx, vals, x, a, b, pre, post, pat, K):
        data = vals.new_zeros(len(pat.rows)).index_add_(0, pat.inverse, vals.reshape(-1))
        ctx.save_for_backward(data, x, a, b, pre, post)
        ctx.pat, ctx.K = pat, K
        A = torch.sparse_csr_tensor(pat.crow, pat.cols, data, (pat.n, pat.n))
        return (A @ x.T.contiguous()).T

    @staticmethod
    def backward(ctx, gy):
        data, x, a, b, pre, post = ctx.saved_tensors
        pat, K = ctx.pat, ctx.K
        g_vals = g_x = None
        if ctx.needs_input_grad[1]:
            At = torch.sparse_csr_tensor(pat.t_crow, pat.t_cols, data[pat.t_order], (pat.n, pat.n))
            g_x = (At @ gy.T.contiguous()).T
        if ctx.needs_input_grad[0]:
            B = x.shape[0]
            X = x.reshape(B, K, -1)[:, :, pre].permute(1, 2, 0).contiguous()  # (K, E, B)
            G = gy.reshape(B, K, -1)[:, :, post].permute(1, 2, 0).contiguous()
            edge = torch.arange(len(pre), device=pre.device)[:, None]
            if a.shape[1] == K * K:
                # all brain pairs: K large elementwise products beat E tiny matmuls
                outer = torch.stack([(X[i] * G).sum(-1) for i in range(K)])  # (K, K, E)
                g_vals = outer[a, b, edge]
            else:
                g_vals = (X[a, edge] * G[b, edge]).sum(-1)
        return g_vals, g_x, None, None, None, None, None, None


def _scaled_hebbian_step(delta, coact, rate):
    """Scale an update by typical co-activity, not by its own spread: weak correlations must give weak updates.
    Normalising each update to unit variance turned noise into full-size weight changes."""
    return rate * delta / (coact.abs().mean() + 1e-8)


def _homeo_activity(model):
    """Per-brain activity (rows, K, N) where the model has several brains, else (rows, N)."""
    brains = getattr(model, "last_activity_brains", None)
    return brains if brains is not None else model.last_activity


@torch.no_grad()
def _record_homeostasis_targets(model):
    model.homeo_target = _homeo_activity(model).abs().mean(0).clone()


@torch.no_grad()
def _homeostatic_update(model, rate=0.3, lo=0.2, hi=5.0):
    """Synaptic scaling: each neuron (in each brain) nudges its input gain back toward its day-0 activity level."""
    current = _homeo_activity(model).abs().mean(0)
    ratio = ((model.homeo_target + 1e-3) / (current + 1e-3)).clamp(0.5, 2.0)
    model.homeo_gain.mul_(ratio ** rate).clamp_(lo, hi)


@torch.no_grad()
def _fast_write(model, modulator=None, row_chunk=64):
    """Decaying Hebbian write to the fast synapse weights, one set per brain. Unlike the slow rule this one cannot
    run away: every write decays first, so a bad adaptation fades instead of being engraved. `modulator` (one value
    per recorded row, rows run time-first) makes it three-factor - and the signal passed in is surprise, measured
    against what actually arrived rather than the model's own confidence."""
    act = _homeo_activity(model)
    if act.dim() == 2:
        act = act[:, None]
    rows, K = act.shape[0], act.shape[1]
    pre, post = model.pre, model.post
    m = None if modulator is None else modulator - modulator.mean()
    delta = torch.zeros_like(model.fast)
    total = torch.zeros((), device=act.device)
    for s in range(0, rows, row_chunk):  # (rows, K, E) at once would be gigabytes
        a = act[s:s + row_chunk]
        co = a[:, :, pre] * a[:, :, post]
        delta += (co if m is None else m[s:s + row_chunk, None, None] * co).sum(0)
        total += co.abs().sum()
    mean_co = total.item() / (rows * K * len(pre)) + 1e-8
    model.fast.mul_(model.fast_decay).add_(model.fast_rate * delta / (rows * mean_co))
    # no synapse may move more than `fast_clamp` (10%) of its strength, so the recurrent gain can only drift by about
    # that much. A heuristic bound, not a guarantee (the strict bound is on |W|'s radius, which can exceed W's) - but
    # v12's [-0.9, 3] allowed 4x synapses and the network ran away
    model.fast.clamp_(-model.fast_clamp, model.fast_clamp)


@torch.no_grad()
def _fast_write_dense(model, modulator=None):
    """The same rule for a model with no connectome: an additive fast weight matrix over the hidden state."""
    act = model.last_activity
    co = act.T @ act / len(act)
    m = None if modulator is None else modulator - modulator.mean()
    delta = co if m is None else (act * m[:, None]).T @ act / len(act)
    model.fast.mul_(model.fast_decay).add_(model.fast_rate * delta / (co.abs().mean() + 1e-8))
    model.fast.clamp_(-model.fast_clamp, model.fast_clamp)


@torch.no_grad()
def _delta_readout(model, probs, target, rate=0.05):
    """Widrow-Hoff on the readout only: post-synaptic error times pre-synaptic activity, local and gradient-free.
    In a language model the next character is a free label, so this error is measured, not guessed - which is
    exactly what every rule that failed under drift was missing."""
    feats = model.last_feats
    err = F.one_hot(target, probs.shape[-1]).float() - probs
    model.readout.weight.add_(rate * (err.T @ feats) / len(feats))
    model.readout.bias.add_(rate * err.mean(0))


def _enable_fast(model, on=True, rate=0.05, decay=0.9, clamp=0.1):
    model.fast_on, model.fast_rate, model.fast_decay, model.fast_clamp = on, rate, decay, clamp
    model.fast.zero_()


def detach_state(state):
    """Recurrent state may be a tensor or, with a fast memory attached, a tuple of tensors."""
    return tuple(s.detach() for s in state) if isinstance(state, tuple) else state.detach()


class FastMemory(nn.Module):
    """Gated delta-rule associative memory trained end to end (the Gated DeltaNet recurrence, Yang et al. 2025).
    At every step the slow weights choose what to store (key k, value v), how hard to write (beta) and how much to
    keep (alpha). The write is error-correcting - it stores v minus what the memory already returns for k - so
    repeated writes converge instead of piling up interference, which is what sank plain Hebbian fast weights.
    Because the update is part of the forward pass, the model keeps learning from a stream with no gradient; and
    because it is trained that way, the rest of the network has learned to use what the memory returns - the step
    v12's bolt-on Hebbian weights never had."""

    def __init__(self, d_state, d_out, d_inj, d_mem=32, d_hid=48):
        super().__init__()
        self.d_mem = d_mem
        self.z = nn.Linear(d_state, d_hid)
        self.qkv = nn.Linear(d_hid, 3 * d_mem, bias=False)
        self.gates = nn.Linear(d_hid, 2)
        self.to_out = nn.Linear(d_mem, d_out, bias=False)
        self.to_inj = nn.Linear(d_mem, d_inj, bias=False)
        with torch.no_grad():
            self.gates.bias.copy_(torch.tensor([3.0, -2.0]))  # start by remembering (alpha ~0.95), writing gently
            self.to_out.weight.zero_()  # the memory starts as a no-op and is learned in
            self.to_inj.weight.zero_()

    def init_state(self, B, device):
        return torch.zeros(B, self.d_mem, self.d_mem, device=device), torch.zeros(B, self.d_mem, device=device)

    def step(self, h, S):
        z = F.silu(self.z(h))
        q, k, v = self.qkv(z).chunk(3, -1)
        q, k = F.normalize(q, dim=-1), F.normalize(k, dim=-1)
        alpha, beta = torch.sigmoid(self.gates(z)).unbind(-1)
        alpha, beta = alpha[:, None, None], beta[:, None, None]
        recalled = torch.bmm(S, k.unsqueeze(-1))  # what the memory returns for this key right now
        S = alpha * (S - beta * recalled @ k.unsqueeze(1)) + beta * v.unsqueeze(-1) @ k.unsqueeze(1)
        return torch.bmm(S, q.unsqueeze(-1)).squeeze(-1), S


def _local_sigma(model, value):
    sigma = float(value) if value else (model.xyz[model.pre] - model.xyz[model.post]).norm(dim=1).median().item()
    return sigma if sigma > 1e-6 else 1.0  # a degenerate geometry would make every locality score NaN


@torch.no_grad()
def _activity_corr(model, u):
    model.forward(u, record=True)
    act = model.last_activity - model.last_activity.mean(0)
    norm = act.norm(dim=0).clamp_min(1e-8)
    return ((act.T @ act) / (norm[:, None] * norm[None, :])).reshape(-1)


@torch.no_grad()
def _rewire(model, step, total_steps, u, opt):
    """Structural plasticity for either brain type: every `rewire_every` steps (until 80% of training) prune the
    weakest synapses and grow as many new ones. Growth rules: random pairs, most correlated pairs (like), pairs sampled
    by distance (local), or correlation penalised by distance (like_local). New synapse signs follow the pre neuron."""
    if step % model.rewire_every or step >= 0.8 * total_steps:
        return
    N, E, dev = model.N, model.E, model.pre.device
    n_swap = int(model.rewire_frac * E)
    log_w = model.log_w.mean(0) if model.log_w.dim() == 2 else model.log_w  # multi-brain: prune by average strength
    strength = (model.w_edge * torch.exp(log_w)).abs()
    slots = torch.topk(strength, n_swap, largest=False).indices
    keep = torch.ones(E, dtype=torch.bool, device=dev)
    keep[slots] = False

    corr = _activity_corr(model, u)
    locality = -((model.xyz[:, None, :] - model.xyz[None, :, :]) ** 2).sum(-1).reshape(-1) / (2 * model.local_sigma ** 2)
    if model.rewire == "random":
        score = torch.rand(N * N, device=dev)
    elif model.rewire == "like":
        score = corr.clone()
    elif model.rewire == "local":
        # Gumbel top-k: samples pairs with probability proportional to exp(locality)
        score = locality - torch.log(-torch.log(torch.rand(N * N, device=dev).clamp_min(1e-12)))
    else:
        score = corr + locality
    score = torch.nan_to_num(score, nan=-float("inf"), posinf=-float("inf"))  # topk ranks NaN as largest
    score[model.pre * N + model.post] = -float("inf")  # existing and just-pruned pairs
    score[torch.arange(N, device=dev) * (N + 1)] = -float("inf")
    new = torch.topk(score, n_swap).indices
    new_pre, new_post = new // N, new % N

    tau_new = model.cls[new_pre] * 2 + model.cls[new_post]
    kept_strength, kept_tau = strength[keep], model.tau[keep]
    typical = torch.stack([kept_strength[kept_tau == t].median() if (kept_tau == t).any() else kept_strength.median()
                           for t in range(N_TYPE_PAIRS)])
    model.pre[slots], model.post[slots], model.tau[slots] = new_pre, new_post, tau_new
    model.w_edge[slots] = typical[tau_new] * (1.0 - 2.0 * model.cls[new_pre].float())
    model.log_w[..., slots] = 0.0
    model.is_original[slots] = False
    state = opt.state.get(model.log_w, {})
    for key in ("exp_avg", "exp_avg_sq"):
        if key in state:
            state[key][..., slots] = 0.0
    model._after_rewire(slots)


def _rigl_scores(model, u, y, loss_fn):
    """RigL (Evci et al. 2020): grow where the loss gradient of a synapse that does not exist yet is largest. A zero
    dense probe added to each brain's recurrent input has exactly that gradient, dL/dP[k, post, pre]. Returned as
    [k, pre * N + post] to match the pair index the growth code uses."""
    assert y is not None and loss_fn is not None, "RigL growth needs the batch targets and the loss"
    probe = torch.zeros(model.K, model.N, model.N, device=u.device, requires_grad=True)
    model._probe = probe
    try:
        with torch.enable_grad():
            grad, = torch.autograd.grad(loss_fn(model(u), y), probe)
    finally:
        model._probe = None
    return grad.abs().transpose(1, 2).reshape(model.K, -1)


@torch.no_grad()
def _grow_scores(model, corr, locality, n_new, live_pairs, grad=None):
    """Pick n_new neuron pairs to connect, by the chosen growth rule, avoiding pairs this brain already has."""
    N, dev = model.N, model.pre.device
    if model.rewire == "rigl":
        score = grad.clone()
    elif model.rewire == "random":
        score = torch.rand(N * N, device=dev)
    elif model.rewire == "like":
        score = corr.clone()
    elif model.rewire == "local":
        score = locality - torch.log(-torch.log(torch.rand(N * N, device=dev).clamp_min(1e-12)))
    else:
        score = corr + locality
    score = torch.nan_to_num(score, nan=-float("inf"), posinf=-float("inf"))  # topk ranks NaN as largest
    score[live_pairs] = -float("inf")
    score[torch.arange(N, device=dev) * (N + 1)] = -float("inf")
    new = torch.topk(score, n_new).indices
    assert bool((new // N != new % N).all()), "growth proposed a self-connection"
    return new // N, new % N


@torch.no_grad()
def _rewire_brains(model, step, total_steps, u, opt, grad=None):
    """Free prune and genesis, one brain at a time. Every brain drops the synapses *it* has made weakest. Where it
    grows depends on the rule: `rigl` uses that brain's own loss gradient, `like`/`like_local` its own activity
    correlations, and `local`/`random` do not depend on the brain at all (distance-weighted or uniform random - the
    SET scheme of Mocanu et al. 2018). v12 ran `local`, so its brains differed by pruning alone."""
    N, dev, S = model.N, model.pre.device, model.slack
    model.forward(u, record=True)
    locality = -((model.xyz[:, None, :] - model.xyz[None, :, :]) ** 2).sum(-1).reshape(-1) / (2 * model.local_sigma ** 2)
    touched = []
    for k in range(model.K):
        live = model.alive[k] > 0
        had = model.pre[live] * N + model.post[live]  # captured before pruning: no prune-and-regrow churn
        n_swap = max(int(model.rewire_frac * int(live.sum())), 1)
        strength = (model.w_edge * torch.exp(model.log_w[k])).abs().masked_fill(~live, float("inf"))
        slots = torch.topk(strength, n_swap, largest=False).indices
        model.alive[k, slots] = 0.0
        private = torch.arange(model.E_real + k * S, model.E_real + (k + 1) * S, device=dev)
        free = private[model.alive[k, private] == 0][:n_swap]
        if not len(free):
            continue
        act = model.last_activity_brains[:, k]  # this brain's own activity decides where it grows
        a = act - act.mean(0)
        corr = ((a.T @ a) / (a.norm(dim=0).clamp_min(1e-8)[:, None] * a.norm(dim=0).clamp_min(1e-8)[None, :])).reshape(-1)
        new_pre, new_post = _grow_scores(model, corr, locality, len(free), had,
                                         None if grad is None else grad[k])
        kept = (model.w_edge * torch.exp(model.log_w[k])).abs()[model.alive[k] > 0]
        typical = kept.median() if len(kept) else model.w_edge.abs().median()
        model.pre[free], model.post[free] = new_pre, new_post
        model.tau[free] = model.cls[new_pre] * 2 + model.cls[new_post]
        model.w_edge[free] = typical * (1.0 - 2.0 * model.cls[new_pre].float())
        # RigL starts new connections at zero; exp(log_w) cannot reach zero, so start small instead
        model.log_w[:, free] = math.log(NEW_SYNAPSE_FRAC)
        model.fast[:, free] = 0.0
        model.is_original[free] = False
        model.alive[k, free] = 1.0
        state = opt.state.get(model.log_w, {}) if opt is not None else {}
        for key in ("exp_avg", "exp_avg_sq"):
            if key in state:
                state[key][:, free] = 0.0
        touched.append(free)
    # a slot no brain uses is retired to (0, 0), where the CSR pattern merges every dead slot into one entry
    dead = (model.alive.sum(0) == 0) & ~model.is_original
    model.pre[dead], model.post[dead], model.w_edge[dead], model.tau[dead] = 0, 0, 0.0, 0
    model._after_rewire(torch.cat(touched) if touched else torch.zeros(0, dtype=torch.long, device=dev))


@torch.no_grad()
def _brain_structure(model):
    """How far the six architectures have actually drifted apart."""
    A = model.alive > 0
    if model.K == 1:
        return {"brain_edges_mean": float(A.sum().item())}
    inter = (A[:, None] & A[None]).sum(-1).float()
    union = (A[:, None] | A[None]).sum(-1).clamp_min(1).float()
    off = ~torch.eye(model.K, dtype=torch.bool, device=A.device)
    # composition over the four (pre, post) excitatory/inhibitory classes: two regions differ in what they wire to what
    comp = torch.stack([torch.bincount(model.tau[A[k]], minlength=N_TYPE_PAIRS).float() for k in range(model.K)])
    comp = comp / comp.sum(1, keepdim=True).clamp_min(1)
    return {"brain_edge_jaccard": (inter / union)[off].mean().item(),
            "brain_edges_mean": A.sum(1).float().mean().item(),
            "brain_edges_spread": A.sum(1).float().std().item(),
            "brain_type_spread": comp.std(0).sum().item(),
            "brain_grown_frac": (A & ~model.is_original[None]).float().sum(1).mean().item() / max(A.sum(1).float().mean().item(), 1)}


@torch.no_grad()
def _wiring_stats(model):
    """Network statistics to compare against MICrONS: reciprocity, connection distance, degree spread, weight spread,
    and (when activity was recorded) how much more correlated connected neurons are than neuron pairs overall."""
    N = model.N
    keys = model.pre * N + model.post
    in_deg = torch.bincount(model.post, minlength=N).float()
    log_w = getattr(model, "log_w", None)
    if log_w is not None and log_w.dim() == 2:
        log_w = log_w.mean(0)
    w = (model.w_edge * torch.exp(log_w) if log_w is not None else model.w_edge).abs()
    s = {
        "reciprocity": torch.isin(model.post * N + model.pre, keys).float().mean().item(),
        "mean_conn_dist": (model.xyz[model.pre] - model.xyz[model.post]).norm(dim=1).mean().item(),
        "in_degree_cv": (in_deg.std() / in_deg.mean()).item(),
        "log_weight_sd": torch.log(w + 1e-12).std().item(),
    }
    if model.last_activity is not None:
        act = model.last_activity - model.last_activity.mean(0)
        norm = act.norm(dim=0).clamp_min(1e-8)
        corr = (act.T @ act) / (norm[:, None] * norm[None, :])
        off = ~torch.eye(N, dtype=torch.bool, device=corr.device)
        s["like_to_like_gap"] = (corr.reshape(-1)[keys].mean() - corr[off].mean()).item()
    return s


class Multiverse(nn.Module):
    """K copies of one connectome. Neurons hold existence amplitudes over brains; every synapse holds a
    two-neuron pair state (pre-brain, post-brain) produced by an entangling gate, and |amplitude|^2 decides
    which brain pairs that synapse links. Modes ablate each ingredient."""

    def __init__(self, conn, d_in, d_out, in_idx, out_idx, K=6, mode="quantum", radius=0.9, plastic=False,
                 rewire="none", rewire_every=100, rewire_frac=0.1, local_sigma=None, per_brain_rewire=False,
                 grow_slack=0.15, dropout=0.0, memory=False):
        super().__init__()
        assert mode in MODES
        self.mode = mode
        self.K = K = 1 if mode == "single" else K
        self.N = N = conn.n

        # copies: rewiring edits these buffers in place and must not touch the shared connectome arrays
        pre = torch.tensor(conn.pre, dtype=torch.long)
        post = torch.tensor(conn.post, dtype=torch.long)
        inh = torch.tensor(conn.inhibitory)
        w = torch.as_tensor(conn.weight, dtype=torch.float32) * (1.0 - 2.0 * inh.float())[pre]
        w = w * (radius / _spectral_radius(conn.pre, conn.post, w.numpy(), N))
        cls = inh.long()
        tau = cls[pre] * 2 + cls[post]
        # per-brain structural plasticity needs somewhere to grow into: give each brain a private pool of spare
        # slots so the edge list never has to resize (which would invalidate the optimizer state on log_w)
        self.E_real = len(pre)
        self.per_brain_rewire = per_brain_rewire
        self.slack = S = int(grow_slack * self.E_real) if per_brain_rewire else 0
        if S:
            pad = torch.zeros(K * S, dtype=torch.long)
            pre, post = torch.cat([pre, pad]), torch.cat([post, pad])
            w = torch.cat([w, torch.zeros(K * S)])
            tau = torch.cat([tau, torch.zeros(K * S, dtype=torch.long)])
        self.E = len(pre)
        self.register_buffer("alive", torch.ones(K, self.E))
        if S:
            self.alive[:, self.E_real:] = 0.0
        self.register_buffer("pre", pre)
        self.register_buffer("post", post)
        self.register_buffer("w_edge", w)
        self.register_buffer("cls", cls)
        self.register_buffer("tau", tau)
        order = torch.argsort(tau)
        self.register_buffer("edge_order", order)
        self.register_buffer("edge_inv", torch.argsort(order))
        self.type_counts = torch.bincount(tau, minlength=N_TYPE_PAIRS).tolist()
        self.register_buffer("in_idx", torch.as_tensor(in_idx, dtype=torch.long))
        self.register_buffer("out_idx", torch.as_tensor(out_idx, dtype=torch.long))
        self.register_buffer("lesion", torch.ones(K, N))
        self.register_buffer("homeo_gain", torch.ones(K, N))
        self.homeo_target = None
        # fast weights: each brain keeps its own short-term modulation of every synapse, written by Hebbian
        # co-activity at deployment time and decaying, so six brains can adapt in six different directions
        self.register_buffer("fast", torch.zeros(K, self.E))
        self.fast_on, self.fast_rate, self.fast_decay, self.fast_clamp = False, 0.05, 0.9, 0.1
        self.last_feats = self.last_brain_feats = None
        self._soft_pattern = None
        self._probe = None  # dense zero matrix whose gradient scores candidate synapses for RigL growth

        routes = torch.arange(K) * (K + 1) if mode in DIAGONAL_MODES else torch.arange(K * K)
        self.register_buffer("routes", routes)
        self.register_buffer("route_a", routes // K)
        self.register_buffer("route_b", routes % K)

        self.log_gain = nn.Parameter(torch.zeros(N_TYPE_PAIRS))
        self.plastic = plastic
        if plastic:
            # per-brain, per-synapse log-strength: every brain remodels its own synapses; exp() keeps E/I signs
            self.log_w = nn.Parameter(torch.zeros(K, self.E))
        assert rewire in REWIRE_RULES
        assert rewire == "none" or plastic, "rewiring needs learnable synapse strengths"
        assert not per_brain_rewire or rewire != "none", "per-brain rewiring needs a growth rule"
        self.rewire, self.rewire_every, self.rewire_frac = rewire, rewire_every, rewire_frac
        self.register_buffer("is_original", torch.arange(self.E) < self.E_real)
        self.register_buffer("xyz", torch.as_tensor(conn.xyz, dtype=torch.float32) / 1000)
        self.local_sigma = _local_sigma(self, local_sigma)
        self.last_activity = None
        self.last_activity_brains = None
        self.faults = None
        self.init_stats = _wiring_stats(self)
        self.leak_logit = nn.Parameter(torch.zeros(2))
        self.bias = nn.Parameter(torch.zeros(N))
        self.inp = nn.Linear(d_in, len(in_idx), bias=False)
        self.readout = nn.Linear(len(out_idx), d_out)
        self.drop = nn.Dropout(dropout)
        self.memory = FastMemory(len(out_idx), d_out, len(in_idx)) if memory else None

        if mode == "separate_matched":
            # capacity control: the 2*K*N parameters the existence amplitudes use, spent on per-brain gains/biases
            self.brain_gain = nn.Parameter(torch.zeros(K, N))
            self.brain_bias = nn.Parameter(torch.zeros(K, N))

        if mode in ("classical", "product", "quantum"):
            self.psi_re = nn.Parameter(1.0 + 0.1 * torch.randn(N, K))
            self.psi_im = nn.Parameter(0.1 * torch.randn(N, K))
        if mode == "classical":
            init = torch.zeros(N_TYPE_PAIRS, K * K)
            init[:, torch.arange(K) * (K + 1)] = 4.0
            self.joint_logits = nn.Parameter(init)
        elif mode == "product":
            for name in ("pre_re", "pre_im", "post_re", "post_im"):
                setattr(self, name, nn.Parameter(0.01 * torch.randn(N_TYPE_PAIRS, K, K)))
        elif mode == "quantum":
            self.gate_re = nn.Parameter(0.01 * torch.randn(N_TYPE_PAIRS, K * K, K * K))
            self.gate_im = nn.Parameter(0.01 * torch.randn(N_TYPE_PAIRS, K * K, K * K))
            self.register_buffer("gate_base", _diagonalizing_reflection(K).to(torch.cfloat))

    def _psi(self):
        psi = torch.complex(self.psi_re, self.psi_im)
        return psi / torch.linalg.vector_norm(psi, dim=1, keepdim=True).clamp_min(1e-8)

    def existence(self):
        if self.mode in ("single",) + DIAGONAL_MODES:
            return self.w_edge.new_full((self.N, self.K), 1.0 / self.K)
        psi = self._psi()
        return psi.real ** 2 + psi.imag ** 2

    def _pair_amplitudes(self):
        K = self.K
        psi = self._psi()
        pair = (psi[self.pre].unsqueeze(2) * psi[self.post].unsqueeze(1)).reshape(-1, K * K)
        if self.mode == "product":
            U = _unitary(self.pre_re, self.pre_im)
            V = _unitary(self.post_re, self.post_im)
            G = torch.einsum("tab,tcd->tacbd", U, V).reshape(N_TYPE_PAIRS, K * K, K * K)
        else:
            G = _unitary(self.gate_re, self.gate_im) @ self.gate_base
        chunks = torch.split(pair[self.edge_order], self.type_counts)
        amp = torch.cat([c @ G[t].transpose(0, 1) for t, c in enumerate(chunks)])
        return amp[self.edge_inv]

    def pair_probs(self):
        """(E, R): probability that each synapse links route r = (pre-brain, post-brain)."""
        K, R = self.K, len(self.routes)
        if self.mode in ("single",) + DIAGONAL_MODES:
            return self.w_edge.new_full((self.E, R), 1.0 / R)
        if self.mode == "classical":
            p = self.existence()
            joint = p[self.pre].unsqueeze(2) * p[self.post].unsqueeze(1)
            q = F.softmax(self.joint_logits, -1).view(N_TYPE_PAIRS, K, K)[self.tau]
            P = (joint * q).reshape(self.E, K * K)
            # floor: existence and joint preferences can point at disjoint brains, underflowing a whole row to 0
            P = P + 1e-9
            return P / P.sum(1, keepdim=True)
        amp = self._pair_amplitudes()
        return amp.real ** 2 + amp.imag ** 2

    def coupling(self, sample=False):
        K, N = self.K, self.N
        P = self.pair_probs()
        gain = torch.exp(self.log_gain)[self.tau][:, None]
        # sqrt(K*P): 6 separate brains reproduce the original weights exactly
        amp = torch.sqrt(K * P + 1e-12)
        if sample:
            # measurement: each synapse is realized K times, routes drawn from P (energy-matched, K*E nonzeros)
            picks = torch.multinomial(P.detach(), K, replacement=True)
            q = amp.gather(1, picks)
            a, b = self.route_a[picks], self.route_b[picks]
            vals = _faulty(gain * self._synapse_weight(b), self.faults) * (1.0 + q - q.detach())
            pat = _Pattern((b * N + self.post[:, None]).reshape(-1), (a * N + self.pre[:, None]).reshape(-1), K * N)
        else:
            a, b = self.route_a.expand(self.E, -1), self.route_b.expand(self.E, -1)
            vals = _faulty(gain * self._synapse_weight(b), self.faults) * amp
            if self._soft_pattern is None or self._soft_pattern.rows.device != vals.device:
                self._soft_pattern = _Pattern((b * N + self.post[:, None]).reshape(-1),
                                              (a * N + self.pre[:, None]).reshape(-1), K * N)
            pat = self._soft_pattern
        return vals, a, b, pat

    def _synapse_weight(self, brain):
        """Signed strength of every synapse as held by the postsynaptic brain, shape like `brain` (E, S).
        Slow strengths come from backprop; the fast term is written online by Hebbian co-activity and decays."""
        w = self.w_edge[:, None]
        if self.plastic or self.fast_on:
            edge = torch.arange(self.E, device=brain.device)[:, None]
            if self.plastic:
                w = w * torch.exp(self.log_w[brain, edge])
            if self.fast_on:
                w = w * (1.0 + self.fast[brain, edge])
        if self.per_brain_rewire:
            w = w * self.alive[brain, torch.arange(self.E, device=brain.device)[:, None]]
        return w if w.shape == brain.shape else w.expand(brain.shape)

    def forward(self, u, sample=False, record=False, state=None, return_state=False):
        B, T, _ = u.shape
        K, N = self.K, self.N
        vals, a, b, pat = self.coupling(sample)
        p = self.existence()
        drive = torch.sqrt(K * p[self.in_idx]).T.unsqueeze(0)
        read = torch.sqrt(p[self.out_idx] / K).T.unsqueeze(0)
        leak = torch.sigmoid(self.leak_logit)[self.cls]
        mem = self.memory
        if isinstance(state, tuple):
            x, S, r = state
        else:
            x = u.new_zeros(B, K, N) if state is None else state
            S, r = mem.init_state(B, u.device) if mem is not None else (None, None)
        outs, acts, fts = [], [], []
        for t in range(T):
            drive_t = self.inp(u[:, t])
            if mem is not None:
                drive_t = drive_t + mem.to_inj(r)  # what the memory recalled last step feeds back in as input
            inj = u.new_zeros(B, K, N).index_copy(2, self.in_idx, drive_t.unsqueeze(1) * drive)
            rec = _MeshMul.apply(vals, x.reshape(B, K * N), a, b, self.pre, self.post, pat, K).reshape(B, K, N)
            if self._probe is not None:
                rec = rec + torch.einsum("bkn,kmn->bkm", x, self._probe)
            pre = rec + inj
            if self.mode == "separate_matched":
                pre = torch.exp(self.brain_gain) * pre + self.brain_bias
            x = (1 - leak) * x + leak * torch.tanh(F.relu(pre * self.homeo_gain + self.bias))
            x = x * self.lesion
            per_brain = x[:, :, self.out_idx] * read
            feats = per_brain.sum(1)
            o = self.readout(self.drop(feats))
            if mem is not None:
                r, S = mem.step(feats, S)
                o = o + mem.to_out(r)
            outs.append(o)
            if record:
                acts.append(x.detach())
                fts.append(per_brain.detach())
        if record:
            self.last_activity_brains = torch.cat(acts)  # (rows, K, N), rows run time-first
            self.last_activity = self.last_activity_brains.mean(1)
            self.last_brain_feats = torch.cat(fts)       # (rows, K, F): one draft per brain
            self.last_feats = self.last_brain_feats.sum(1)
        out = torch.stack(outs, 1)
        final = (x, S, r) if mem is not None else x
        return (out, final) if return_state else out

    @torch.no_grad()
    def set_lesion(self, neurons=None, brains=None):
        self.lesion.fill_(1.0)
        if neurons is not None:
            self.lesion[:, torch.as_tensor(neurons, device=self.lesion.device)] = 0.0
        if brains is not None:
            self.lesion[brains, :] = 0.0

    def maybe_rewire(self, step, total_steps, u, opt, y=None, loss_fn=None):
        if not self.per_brain_rewire:
            return _rewire(self, step, total_steps, u, opt)
        if step % self.rewire_every or step >= 0.8 * total_steps:
            return
        grad = _rigl_scores(self, u, y, loss_fn) if self.rewire == "rigl" else None
        _rewire_brains(self, step, total_steps, u, opt, grad)

    def _after_rewire(self, slots):
        order = torch.argsort(self.tau)
        self.edge_order.copy_(order)
        self.edge_inv.copy_(torch.argsort(order))
        self.type_counts = torch.bincount(self.tau, minlength=N_TYPE_PAIRS).tolist()
        self._soft_pattern = None

    def set_faults(self, **faults):
        self.faults = faults or None

    @torch.no_grad()
    def local_update(self, rate=0.05, decay=0.5, clamp=2.0, modulator=None):
        """Gradient-free learning on synapse strengths; signs never change. Plain: co-activity with Oja-style decay.
        Three-factor (modulator given, one value per recorded row): co-activity is strengthened only when it
        coincides with an above-average third signal (e.g. the model's own confidence) and weakened when below."""
        if self.last_activity_brains is None or not self.plastic:
            return
        for k in range(self.K):  # each brain learns from its own activity
            act = self.last_activity_brains[:, k]
            coact = act[:, self.pre] * act[:, self.post]
            if modulator is None:
                delta = coact.mean(0) - decay * (act[:, self.post] ** 2).mean(0) * torch.exp(self.log_w[k])
            else:
                delta = ((modulator - modulator.mean())[:, None] * coact).mean(0)
            self.log_w[k] += _scaled_hebbian_step(delta, coact, rate)
        self.log_w.clamp_(-clamp, clamp)

    def record_homeostasis_targets(self):
        _record_homeostasis_targets(self)

    def homeostatic_update(self, rate=0.3):
        _homeostatic_update(self, rate)

    def enable_fast(self, on=True, rate=0.05, decay=0.9, clamp=0.1):
        _enable_fast(self, on, rate, decay, clamp)

    def delta_readout_update(self, probs, target, rate=0.05):
        _delta_readout(self, probs, target, rate)

    def fast_write(self, modulator=None):
        _fast_write(self, modulator)

    @torch.no_grad()
    def brain_logits(self):
        """One output distribution per brain: the competing drafts behind the collapsed answer."""
        return self.readout(self.last_brain_feats * self.K)  # (rows, K, V)

    @torch.no_grad()
    def diagnostics(self):
        d = {"params": sum(p.numel() for p in self.parameters())}
        if self.fast_on:
            d["fast_abs_mean"] = self.fast.abs().mean().item()
            d["fast_brain_spread"] = (self.fast.std(0).mean() / (self.fast.abs().mean() + 1e-8)).item()
        if self.per_brain_rewire:
            d.update(_brain_structure(self))
        if self.plastic:
            d["synapse_log_change"] = self.log_w.abs().mean().item()
            d["synapses_changed_2x"] = (self.log_w.abs() > math.log(2)).float().mean().item()
        if self.rewire != "none":
            d["original_synapses_kept"] = self.is_original.float().mean().item()
        d.update({f"init_{k}": v for k, v in self.init_stats.items()})
        d.update(_wiring_stats(self))
        K = self.K
        if K == 1:
            return d
        P = self.w_edge.new_zeros(self.E, K * K)
        P[:, self.routes] = self.pair_probs()
        P = P.view(-1, K, K)
        pa, pb = P.sum(2, keepdim=True), P.sum(1, keepdim=True)
        mi = (P * (torch.log(P + 1e-12) - torch.log(pa * pb + 1e-12))).sum((1, 2))
        d["same_brain_frac"] = P.diagonal(dim1=1, dim2=2).sum(1).mean().item()
        d["brain_mi_bits"] = (mi.mean() / math.log(2)).item()
        p = self.existence()
        d["existence_entropy"] = (-(p * torch.log(p + 1e-12)).sum(1) / math.log(K)).mean().item()
        if self.mode in ("product", "quantum"):
            s2 = torch.linalg.svdvals(self._pair_amplitudes().view(-1, K, K)) ** 2
            d["entanglement_bits"] = (-(s2 * torch.log2(s2 + 1e-12)).sum(1)).mean().item()
        return d


BASELINE_MODES = ("gru", "transformer")


class GRUBrain(nn.Module):
    """Standard recurrent baseline with no connectome, same task interface (lesions, faults, diagnostics)."""

    def __init__(self, d_in, d_out, hidden=512, dropout=0.0, memory=False):
        super().__init__()
        self.K, self.N, self.mode, self.rewire = 1, hidden, "gru", "none"
        self.cell = nn.GRUCell(d_in, hidden)
        self.readout = nn.Linear(hidden, d_out)
        self.drop = nn.Dropout(dropout)
        self.memory = FastMemory(hidden, d_out, 3 * hidden) if memory else None
        self.register_buffer("lesion", torch.ones(1, hidden))
        self.register_buffer("homeo_gain", torch.ones(hidden))
        self.homeo_target = None
        # dense additive fast weights (Ba-style), the no-connectome counterpart of the per-synapse version
        self.register_buffer("fast", torch.zeros(hidden, hidden))
        self.fast_on, self.fast_rate, self.fast_decay, self.fast_clamp = False, 0.05, 0.9, 0.1
        self.faults = None
        self.last_activity = self.last_feats = None

    def forward(self, u, sample=False, record=False, state=None, return_state=False):
        B, T, _ = u.shape
        w_hh = self.cell.weight_hh
        if self.faults:
            w_hh = _faulty(w_hh.reshape(-1), self.faults).reshape_as(w_hh)
        mem = self.memory
        if isinstance(state, tuple):
            h, S, rd = state
        else:
            h = u.new_zeros(B, self.N) if state is None else state
            S, rd = mem.init_state(B, u.device) if mem is not None else (None, None)
        outs, acts = [], []
        for t in range(T):
            gi = F.linear(u[:, t], self.cell.weight_ih, self.cell.bias_ih) * self.homeo_gain.repeat(3)
            if mem is not None:
                gi = gi + mem.to_inj(rd)
            i_r, i_z, i_n = gi.chunk(3, 1)
            h_r, h_z, h_n = F.linear(h, w_hh, self.cell.bias_hh).chunk(3, 1)
            if self.fast_on:
                h_n = h_n + h @ self.fast.T
            r, z = torch.sigmoid(i_r + h_r), torch.sigmoid(i_z + h_z)
            h = (1 - z) * torch.tanh(i_n + r * h_n) + z * h
            h = h * self.lesion
            o = self.readout(self.drop(h))
            if mem is not None:
                rd, S = mem.step(h, S)
                o = o + mem.to_out(rd)
            outs.append(o)
            if record:
                acts.append(h.detach())
        if record:
            self.last_activity = self.last_feats = torch.cat(acts)
        out = torch.stack(outs, 1)
        final = (h, S, rd) if mem is not None else h
        return (out, final) if return_state else out

    @torch.no_grad()
    def set_lesion(self, neurons=None, brains=None):
        self.lesion.fill_(1.0)
        if neurons is not None:
            self.lesion[:, torch.as_tensor(neurons, device=self.lesion.device)] = 0.0

    def set_faults(self, **faults):
        self.faults = faults or None

    @torch.no_grad()
    def local_update(self, rate=0.05, decay=0.5, clamp=2.0, modulator=None):
        """Plain or three-factor Hebbian update on the recurrent weights, gradient-free, matched to the brains' rule."""
        if self.last_activity is None:
            return
        act = self.last_activity
        if modulator is None:
            delta = act.T @ act / len(act) - decay * (act ** 2).mean(0)[:, None] * self.cell.weight_hh[:self.N]
        else:
            delta = (act * (modulator - modulator.mean())[:, None]).T @ act / len(act)
        self.cell.weight_hh[:self.N] += _scaled_hebbian_step(delta, act.T @ act / len(act), rate)
        self.cell.weight_hh.clamp_(-clamp, clamp)

    def record_homeostasis_targets(self):
        _record_homeostasis_targets(self)

    def homeostatic_update(self, rate=0.3):
        _homeostatic_update(self, rate)

    def enable_fast(self, on=True, rate=0.05, decay=0.9, clamp=0.1):
        _enable_fast(self, on, rate, decay, clamp)

    def delta_readout_update(self, probs, target, rate=0.05):
        _delta_readout(self, probs, target, rate)

    def fast_write(self, modulator=None):
        _fast_write_dense(self, modulator)

    @torch.no_grad()
    def diagnostics(self):
        d = {"params": sum(p.numel() for p in self.parameters())}
        if self.fast_on:
            d["fast_abs_mean"] = self.fast.abs().mean().item()
        return d


class TransformerLM(nn.Module):
    """Small causal character transformer: the reference for what this corpus supports at a given parameter count.
    Attention needs the characters themselves, so its 'state' is the recent context, capped at `block`."""

    def __init__(self, d_in, d_out, d_model=64, n_layer=3, n_head=4, block=64, dropout=0.0):
        super().__init__()
        self.K, self.N, self.mode, self.rewire, self.block = 1, d_model, "transformer", "none", block
        self.emb = nn.Linear(d_in, d_model, bias=False)  # inputs are one-hot, so this is an embedding table
        self.pos = nn.Parameter(0.02 * torch.randn(block, d_model))
        layer = nn.TransformerEncoderLayer(d_model, n_head, 4 * d_model, dropout, activation="gelu",
                                           batch_first=True, norm_first=True)
        self.blocks = nn.TransformerEncoder(layer, n_layer, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
        self.readout = nn.Linear(d_model, d_out)
        self.register_buffer("lesion", torch.ones(1, d_model))
        self.last_activity = self.last_feats = None
        self.memory = None

    def forward(self, u, sample=False, record=False, state=None, return_state=False):
        T = u.shape[1]
        assert T <= self.block, "feed at most `block` new characters per call"
        ctx = (u if state is None else torch.cat([state, u], 1))[:, -self.block:]
        L = ctx.shape[1]
        mask = torch.triu(torch.full((L, L), float("-inf"), device=u.device), 1)
        h = self.norm(self.blocks(self.drop(self.emb(ctx) + self.pos[:L]), mask=mask, is_causal=True))[:, -T:]
        out = self.readout(self.drop(h))
        if record:
            self.last_activity = self.last_feats = h.detach().transpose(0, 1).reshape(-1, h.shape[-1])
        return (out, ctx.detach()) if return_state else out

    def set_lesion(self, neurons=None, brains=None):
        pass

    def delta_readout_update(self, probs, target, rate=0.05):
        _delta_readout(self, probs, target, rate)

    @torch.no_grad()
    def diagnostics(self):
        return {"params": sum(p.numel() for p in self.parameters())}


PHASE_MODES = ("complex_frozen", "complex_randphase", "complex_phase")


class PhaseBrain(nn.Module):
    """One connectome brain with complex-valued activity. Synapses carry phases in (-90, 90) degrees, so the real part
    keeps its excitatory/inhibitory sign while signals along different paths can interfere; readout sees Re z, Im z, |z|^2.
    complex_phase learns phases (strengths fixed); complex_frozen fixes phases at 0 and learns strengths instead,
    so both add exactly one parameter per synapse."""

    def __init__(self, conn, d_in, d_out, in_idx, out_idx, mode="complex_phase", radius=0.9,
                 rewire="none", rewire_every=100, rewire_frac=0.1, local_sigma=None):
        super().__init__()
        assert mode in PHASE_MODES
        self.mode, self.K, self.N, self.E = mode, 1, conn.n, len(conn.pre)
        N = conn.n
        # copies: rewiring edits these buffers in place and must not touch the shared connectome arrays
        pre = torch.tensor(conn.pre, dtype=torch.long)
        post = torch.tensor(conn.post, dtype=torch.long)
        inh = torch.tensor(conn.inhibitory)
        w = torch.as_tensor(conn.weight, dtype=torch.float32) * (1.0 - 2.0 * inh.float())[pre]
        w = w * (radius / _spectral_radius(conn.pre, conn.post, w.numpy(), N))
        cls = inh.long()
        for name, t in (("pre", pre), ("post", post), ("w_edge", w), ("cls", cls), ("tau", cls[pre] * 2 + cls[post])):
            self.register_buffer(name, t)
        self.register_buffer("in_idx", torch.as_tensor(in_idx, dtype=torch.long))
        self.register_buffer("out_idx", torch.as_tensor(out_idx, dtype=torch.long))
        self.register_buffer("lesion", torch.ones(1, N))
        self.register_buffer("homeo_gain", torch.ones(N))
        self.homeo_target = None
        self.register_buffer("fast", torch.zeros(1, self.E))
        self.fast_on, self.fast_rate, self.fast_decay, self.fast_clamp = False, 0.05, 0.9, 0.1
        self.last_feats = None
        self.register_buffer("route0", torch.zeros(self.E, 1, dtype=torch.long))
        self._pattern = None

        self.log_gain = nn.Parameter(torch.zeros(N_TYPE_PAIRS))
        self.leak_logit = nn.Parameter(torch.zeros(2))
        self.bias = nn.Parameter(torch.zeros(N))
        self.inp = nn.Linear(d_in, len(in_idx), bias=False)
        # reads Re z, Im z and |z|^2: |z|^2 alone is blind to the input's sign, since flipping it maps z -> -z
        self.readout = nn.Linear(3 * len(out_idx), d_out)
        if mode == "complex_phase":
            # all-zero phases keep every imaginary part at 0, where the |z|^2 readout gives phases exactly zero gradient
            self.phase_raw = nn.Parameter(0.4 * torch.randn(self.E))
        else:
            if mode == "complex_randphase":
                # same random draw as complex_phase's init but frozen: separates having phases from learning them
                self.register_buffer("phase_fixed", (math.pi / 2) * torch.tanh(0.4 * torch.randn(self.E)))
            self.log_w = nn.Parameter(torch.zeros(self.E))
        assert rewire in REWIRE_RULES
        assert rewire == "none" or mode != "complex_phase", "rewiring needs learnable synapse strengths"
        self.rewire, self.rewire_every, self.rewire_frac = rewire, rewire_every, rewire_frac
        self.register_buffer("is_original", torch.ones(self.E, dtype=torch.bool))
        self.register_buffer("xyz", torch.as_tensor(conn.xyz, dtype=torch.float32) / 1000)
        self.local_sigma = _local_sigma(self, local_sigma)
        self.last_activity = None
        self.last_activity_brains = None
        self.faults = None
        self.init_stats = _wiring_stats(self)

    def phases(self):
        if self.mode == "complex_phase":
            return (math.pi / 2) * torch.tanh(self.phase_raw)
        if self.mode == "complex_randphase":
            return self.phase_fixed
        return self.w_edge.new_zeros(self.E)

    def forward(self, u, sample=False, record=False, state=None, return_state=False):
        B, T, _ = u.shape
        N = self.N
        w = self.w_edge * torch.exp(self.log_gain)[self.tau]
        if self.mode != "complex_phase":
            w = w * torch.exp(self.log_w)
        if self.fast_on:
            w = w * (1.0 + self.fast[0])
        w = _faulty(w, self.faults)
        theta = self.phases()
        if self.faults and self.faults.get("phase_noise_deg"):
            g = torch.Generator().manual_seed(self.faults.get("seed", 0) + 1)
            theta = theta + math.radians(self.faults["phase_noise_deg"]) * torch.randn(self.E, generator=g).to(theta.device)
        j_re, j_im = (w * torch.cos(theta)).unsqueeze(1), (w * torch.sin(theta)).unsqueeze(1)
        if self._pattern is None or self._pattern.rows.device != w.device:
            self._pattern = _Pattern(self.post, self.pre, N)
        pat, r0 = self._pattern, self.route0
        leak = torch.sigmoid(self.leak_logit)[self.cls]
        z = u.new_zeros(2 * B, N) if state is None else state  # rows 0..B-1 real, rows B..2B-1 imaginary
        outs, acts, fts = [], [], []
        for t in range(T):
            a = _MeshMul.apply(j_re, z, r0, r0, self.pre, self.post, pat, 1)
            b = _MeshMul.apply(j_im, z, r0, r0, self.pre, self.post, pat, 1)
            inj = u.new_zeros(B, N).index_copy(1, self.in_idx, self.inp(u[:, t]))
            v_re = (a[:B] - b[B:] + inj) * self.homeo_gain
            v_im = (b[:B] + a[B:]) * self.homeo_gain
            mag = torch.sqrt(v_re ** 2 + v_im ** 2 + 1e-8)
            scale = torch.tanh(F.relu(mag + self.bias)) / mag
            z = (1 - leak) * z + leak * torch.cat([v_re * scale, v_im * scale])
            z = z * self.lesion
            re, im = z[:B, self.out_idx], z[B:, self.out_idx]
            feats = torch.cat([re, im, re ** 2 + im ** 2], 1)
            outs.append(self.readout(feats))
            if record:
                acts.append(torch.sqrt(z[:B] ** 2 + z[B:] ** 2).detach())
                fts.append(feats.detach())
        if record:
            self.last_activity = torch.cat(acts)
            self.last_feats = torch.cat(fts)
        out = torch.stack(outs, 1)
        return (out, z) if return_state else out

    @torch.no_grad()
    def set_lesion(self, neurons=None, brains=None):
        self.lesion.fill_(1.0)
        if neurons is not None:
            self.lesion[:, torch.as_tensor(neurons, device=self.lesion.device)] = 0.0

    def maybe_rewire(self, step, total_steps, u, opt, y=None, loss_fn=None):
        _rewire(self, step, total_steps, u, opt)

    def _after_rewire(self, slots):
        self._pattern = None
        if self.mode == "complex_randphase":
            self.phase_fixed[slots] = (math.pi / 2) * torch.tanh(0.4 * torch.randn(len(slots), device=slots.device))

    def set_faults(self, **faults):
        self.faults = faults or None

    @torch.no_grad()
    def local_update(self, rate=0.05, decay=0.5, clamp=2.0, modulator=None):
        """Gradient-free plain or three-factor update on synapse strengths, using the recorded activity magnitude."""
        if self.last_activity is None or self.mode == "complex_phase":
            return
        act = self.last_activity
        coact = act[:, self.pre] * act[:, self.post]
        if modulator is None:
            delta = coact.mean(0) - decay * (act[:, self.post] ** 2).mean(0) * torch.exp(self.log_w)
        else:
            delta = ((modulator - modulator.mean())[:, None] * coact).mean(0)
        self.log_w += _scaled_hebbian_step(delta, coact, rate)
        self.log_w.clamp_(-clamp, clamp)

    def record_homeostasis_targets(self):
        _record_homeostasis_targets(self)

    def homeostatic_update(self, rate=0.3):
        _homeostatic_update(self, rate)

    def enable_fast(self, on=True, rate=0.05, decay=0.9, clamp=0.1):
        _enable_fast(self, on, rate, decay, clamp)

    def delta_readout_update(self, probs, target, rate=0.05):
        _delta_readout(self, probs, target, rate)

    def fast_write(self, modulator=None):
        _fast_write(self, modulator)

    @torch.no_grad()
    def diagnostics(self):
        d = {"params": sum(p.numel() for p in self.parameters())}
        if self.mode == "complex_phase":
            deg = self.phases().abs() * 180 / math.pi
            d["mean_abs_phase_deg"] = deg.mean().item()
            d["synapses_phase_gt45"] = (deg > 45).float().mean().item()
        else:
            d["synapse_log_change"] = self.log_w.abs().mean().item()
            d["synapses_changed_2x"] = (self.log_w.abs() > math.log(2)).float().mean().item()
        if self.rewire != "none":
            d["original_synapses_kept"] = self.is_original.float().mean().item()
        d.update({f"init_{k}": v for k, v in self.init_stats.items()})
        d.update(_wiring_stats(self))
        return d

In [ ]:
%%writefile /kaggle/working/multiverse/tasks.py
import copy
import math
import os
import time

import numpy as np
import torch
import torch.nn.functional as F
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from model import detach_state


def _device(model):
    return next(model.parameters()).device


@torch.no_grad()
def _diagnose(model, probe):
    model.forward(probe.to(_device(model)), record=True)
    return model.diagnostics()


HARDWARE_FAULTS = {
    "wnoise10": dict(weight_noise=0.1), "wnoise30": dict(weight_noise=0.3),
    "dead10": dict(dead_frac=0.1), "dead30": dict(dead_frac=0.3),
    "bits8": dict(bits=8), "bits4": dict(bits=4),
}
PHASE_FAULTS = {"phase10": dict(phase_noise_deg=10), "phase30": dict(phase_noise_deg=30)}


JOINT_LR_MULT = 10.0  # classical routing has 144 logits vs quantum's ~10k gate entries; at 1x they barely move


SPLIT = "test"  # run.py: "val" while tuning hyperparameters, "test" only for the reported run
WEIGHT_DECAY = 0.0  # run.py sets it per condition
WARMUP = 100
EVAL_LIMIT = int(os.environ.get("EVAL_LIMIT", "0"))  # smoke tests only: cap scored windows; 0 = whole split
GEN_CHARS = int(os.environ.get("GEN_CHARS", "2000"))  # smoke tests only: shorter samples


def _split_bounds(n, train_end=0.8, val_end=0.9):
    """Contiguous train / val / test parts: returns the train range and whichever of val or test SPLIT selects.
    Nothing reported is evaluated on data that was used to choose hyperparameters."""
    a, b = int(train_end * n), int(val_end * n)
    return (0, a), ((a, b) if SPLIT == "val" else (b, n))


def fit(model, make_batch, loss_fn, steps, lr=3e-3, sample_prob=0.0, checkpoint_every=0, on_checkpoint=None,
        stateful=False):
    """AdamW with linear warmup and cosine decay to 10%, the same recipe for every model. Weight decay applies to
    weight matrices only; for the brains that includes the per-synapse log-strengths, so it pulls them back toward
    the measured connectome. `stateful` carries the recurrent state across batches (truncated backprop through time)."""
    device = _device(model)
    named = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    groups = [{"params": [p for n, p in named if n != "joint_logits" and p.dim() >= 2], "weight_decay": WEIGHT_DECAY},
              {"params": [p for n, p in named if n != "joint_logits" and p.dim() < 2], "weight_decay": 0.0}]
    joint = [p for n, p in named if n == "joint_logits"]
    if joint:
        groups.append({"params": joint, "weight_decay": 0.0, "lr_mult": JOINT_LR_MULT})
    opt = torch.optim.AdamW([g for g in groups if g["params"]], lr=lr)
    model.train()
    t0 = time.time()
    state = None
    for step in range(steps):
        scale = min(1.0, (step + 1) / WARMUP) * (0.1 + 0.45 * (1 + math.cos(math.pi * step / max(steps, 1))))
        for g in opt.param_groups:
            g["lr"] = lr * scale * g.get("lr_mult", 1.0)
        u, y = make_batch()
        u, y = u.to(device), y.to(device)
        sample = bool(torch.rand(()) < sample_prob)
        if stateful:
            out, state = model(u, sample=sample, state=state, return_state=True)
            state = detach_state(state)
        else:
            out = model(u, sample=sample)
        loss = loss_fn(out, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        if getattr(model, "rewire", "none") != "none":
            model.maybe_rewire(step + 1, steps, u, opt, y=y, loss_fn=loss_fn)
        if checkpoint_every and (step + 1) % checkpoint_every == 0:
            model.eval()
            on_checkpoint(step + 1)
            model.train()
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    model.eval()
    return (time.time() - t0) / max(steps, 1)


def memory(model_fn, steps, sample_prob, lr=3e-3,D=20, T=60, B=32):
    model = model_fn(d_in=1, d_out=D)

    def batch(bs=B):
        u = torch.rand(bs, T, 1) * 2 - 1
        y = torch.stack([torch.roll(u[..., 0], d, dims=1) for d in range(1, D + 1)], -1)
        return u, y

    sec = fit(model, batch, lambda o, y: F.mse_loss(o[:, D:], y[:, D:]), steps, lr=lr, sample_prob=sample_prob)
    u, y = batch(256)
    with torch.no_grad():
        o = model(u.to(_device(model)))[:, D:].reshape(-1, D).cpu()
    t = y[:, D:].reshape(-1, D)
    r2 = [torch.corrcoef(torch.stack([o[:, d], t[:, d]]))[0, 1].nan_to_num().item() ** 2 for d in range(D)]
    return {"capacity": sum(r2), "sec_per_step": sec, **model.diagnostics()}


def _narma10(bs, T):
    u = torch.rand(bs, T) * 0.5
    y = torch.zeros(bs, T)
    for t in range(9, T - 1):
        y[:, t + 1] = 0.3 * y[:, t] + 0.05 * y[:, t] * y[:, t - 9:t + 1].sum(1) + 1.5 * u[:, t - 9] * u[:, t] + 0.1
    # NARMA-10 occasionally diverges (~1% of batches reach inf); redraw those series
    bad = ~(torch.isfinite(y).all(1) & (y.abs().max(1).values < 1.5))
    if bad.any():
        u_new, y_new = _narma10(int(bad.sum()), T)
        u[bad], y[bad] = u_new[..., 0], y_new[..., 0]
    return u.unsqueeze(-1), y.unsqueeze(-1)


def narma10(model_fn, steps, sample_prob, lr=3e-3,T=100, washout=30, B=32):
    model = model_fn(d_in=1, d_out=1)
    loss = lambda o, y: F.mse_loss(o[:, washout:-1], y[:, washout + 1:])
    sec = fit(model, lambda: _narma10(B, T), loss, steps, lr=lr, sample_prob=sample_prob)
    u, y = _narma10(256, T)
    with torch.no_grad():
        o = model(u.to(_device(model))).cpu()
    target = y[:, washout + 1:]
    nrmse = torch.sqrt(F.mse_loss(o[:, washout:-1], target) / target.var()).item()
    return {"nrmse": nrmse, "sec_per_step": sec, **model.diagnostics()}


def _digits():
    d = load_digits()
    X = torch.tensor(d.images / 16.0, dtype=torch.float32)
    y = torch.tensor(d.target)
    perm = torch.randperm(len(y), generator=torch.Generator().manual_seed(0))
    split = int(0.8 * len(y))
    return X[perm[:split]], y[perm[:split]], X[perm[split:]], y[perm[split:]]


def _as_sequence(img, blank=4):
    return torch.cat([img, img.new_zeros(img.shape[0], blank, 8)], 1)


@torch.no_grad()
def _probs(model, X, sample=False):
    return F.softmax(model(_as_sequence(X).to(_device(model)), sample=sample)[:, -1], -1).cpu()


def _acc(probs, y):
    return (probs.argmax(1) == y).float().mean().item()


def _ece(probs, y, bins=15):
    conf, pred = probs.max(1)
    correct = (pred == y).float()
    edges = torch.linspace(0, 1, bins + 1)
    total = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            total += m.float().mean().item() * abs(conf[m].mean().item() - correct[m].mean().item())
    return total


def _entropy(p):
    return -(p * torch.log(p + 1e-12)).sum(1)


def digits(model_fn, steps, sample_prob, lr=3e-3,B=64, universes=8):
    Xtr, ytr, Xte, yte = _digits()
    model = model_fn(d_in=8, d_out=10)

    def batch():
        i = torch.randint(len(ytr), (B,))
        return _as_sequence(Xtr[i]), ytr[i]

    sec = fit(model, batch, lambda o, y: F.cross_entropy(o[:, -1], y), steps, lr=lr, sample_prob=sample_prob)
    res = {"sec_per_step": sec}
    probs = _probs(model, Xte)
    res["acc"] = _acc(probs, yte)
    res["ece"] = _ece(probs, yte)
    for sigma in (0.3, 0.6):
        res[f"acc_noise{sigma}"] = _acc(_probs(model, (Xte + sigma * torch.randn_like(Xte)).clamp(0, 1)), yte)

    ood = Xte.reshape(-1, 64).gather(1, torch.argsort(torch.rand(len(Xte), 64), 1)).reshape(-1, 8, 8)
    labels = [0] * len(Xte) + [1] * len(ood)
    soft_scores = torch.cat([_entropy(_probs(model, Xte)), _entropy(_probs(model, ood))])
    res["ood_auroc_soft"] = roc_auc_score(labels, soft_scores.numpy())

    def universe_entropy(X):
        return _entropy(torch.stack([_probs(model, X, sample=True) for _ in range(universes)]).mean(0))

    uni_scores = torch.cat([universe_entropy(Xte), universe_entropy(ood)])
    res["ood_auroc_universes"] = roc_auc_score(labels, uni_scores.numpy())
    res["acc_universes"] = _acc(torch.stack([_probs(model, Xte, sample=True) for _ in range(universes)]).mean(0), yte)

    for frac in (0.1, 0.3, 0.5):
        model.set_lesion(neurons=torch.randperm(model.N)[:int(frac * model.N)])
        res[f"acc_lesion{int(frac * 100)}"] = _acc(_probs(model, Xte), yte)
    if model.K > 1:
        model.set_lesion(brains=[0])
        res["acc_lose_one_brain"] = _acc(_probs(model, Xte), yte)
    model.set_lesion()
    faults = dict(HARDWARE_FAULTS, **(PHASE_FAULTS if hasattr(model, "phases") else {}))
    for name, f in faults.items():
        model.set_faults(**f)
        res[f"acc_{name}"] = _acc(_probs(model, Xte), yte)
    model.set_faults()
    res.update(_diagnose(model, _as_sequence(Xte[:256])))
    return res


def decision(model_fn, steps, sample_prob, lr=3e-3,T=30, B=64, cohs=(0.0, 0.05, 0.1, 0.2, 0.4)):
    model = model_fn(d_in=1, d_out=2)

    def batch(bs=B, coh=None):
        s = torch.randint(0, 2, (bs,))
        c = torch.tensor(cohs)[torch.randint(len(cohs), (bs,))] if coh is None else torch.full((bs,), coh)
        return (c * (2 * s - 1)).view(bs, 1, 1) + torch.randn(bs, T, 1), s

    sec = fit(model, batch, lambda o, y: F.cross_entropy(o[:, -1], y), steps, lr=lr, sample_prob=sample_prob)
    res = {"sec_per_step": sec}
    ratios = []
    for c in cohs:
        u, s = batch(1000, c)
        with torch.no_grad():
            acc = (model(u.to(_device(model)))[:, -1].argmax(1).cpu() == s).float().mean().item()
        res[f"acc_coh{c}"] = acc
        if c > 0:
            ratios.append(acc / (0.5 * (1 + math.erf(c * math.sqrt(T) / math.sqrt(2)))))
    res["frac_of_ideal"] = sum(ratios) / len(ratios)
    res.update(model.diagnostics())
    return res


def continual(model_fn, steps, sample_prob, lr=3e-3,B=64):
    Xtr, ytr, Xte, yte = _digits()
    model = model_fn(d_in=8, d_out=2)
    pairs = [(2 * k, 2 * k + 1) for k in range(5)]

    def subset(X, y, a, b):
        m = (y == a) | (y == b)
        return X[m], (y[m] == b).long()

    tests = [subset(Xte, yte, a, b) for a, b in pairs]
    history = []
    for a, b in pairs:
        X, Y = subset(Xtr, ytr, a, b)

        def batch():
            i = torch.randint(len(Y), (B,))
            return _as_sequence(X[i]), Y[i]

        fit(model, batch, lambda o, y: F.cross_entropy(o[:, -1], y), steps, lr=lr, sample_prob=sample_prob)
        history.append([_acc(_probs(model, Xt), Yt) for Xt, Yt in tests])
    final = history[-1]
    forgetting = [max(history[i][j] for i in range(j, len(pairs))) - final[j] for j in range(len(pairs) - 1)]
    res = {"final_avg_acc": sum(final) / len(final), "avg_forgetting": sum(forgetting) / len(forgetting)}
    res.update(_diagnose(model, _as_sequence(Xte[:128])))
    return res


TEXT_PATH = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "char_lm", "tinyshakespeare.txt")
_CORPUS = None
_JUDGE = None


def _corpus():
    """Raw text, vocabulary, char->index map and the index tensor, loaded once and shared by text and stream."""
    global _CORPUS
    if _CORPUS is None:
        raw = open(TEXT_PATH, encoding="utf-8").read()
        chars = sorted(set(raw))
        stoi = {c: i for i, c in enumerate(chars)}
        _CORPUS = raw, chars, stoi, torch.tensor([stoi[c] for c in raw])
    return _CORPUS


def _windows(src, T):
    """Every non-overlapping length-T window of `src` with its next-character targets. Evaluation is deterministic,
    so two models are always scored on exactly the same characters (v1-v12 drew random batches)."""
    n = (len(src) - 1) // T
    seq = src[torch.arange(n)[:, None] * T + torch.arange(T + 1)[None, :]]
    return seq[:, :-1], seq[:, 1:]


@torch.no_grad()
def _eval_nll(model, x, y, V, bs=256):
    """Bits for every scored character, batch-first rows (window-major), aligned with y.reshape(-1)."""
    device, out = _device(model), []
    for i in range(0, len(x), bs):
        o = model(F.one_hot(x[i:i + bs], V).float().to(device))
        out.append(F.cross_entropy(o.reshape(-1, V), y[i:i + bs].reshape(-1).to(device), reduction="none").cpu())
    return torch.cat(out) / math.log(2)


def _ngrams(text, n):
    return [text[i:i + n] for i in range(len(text) - n + 1)]


class _CharJudge:
    """Witten-Bell interpolated character 5-gram model of the training text: a fixed judge of how corpus-like a
    generated sample is, identical for every condition and independent of the models being judged. It is not MAUVE
    (no embedding model is available offline) and it can be fooled by repetition, so it is read next to distinct_4."""

    def __init__(self, text, order=5):
        self.order, self.V, self.tables = order, len(set(text)), []
        for n in range(order):  # context length n
            counts = {}
            for i in range(n, len(text)):
                d = counts.setdefault(text[i - n:i], {})
                d[text[i]] = d.get(text[i], 0) + 1
            self.tables.append({ctx: (d, sum(d.values()), len(d)) for ctx, d in counts.items()})

    def prob(self, hist, ch):
        p = 1.0 / self.V
        for n in range(min(self.order, len(hist) + 1)):
            entry = self.tables[n].get(hist[len(hist) - n:] if n else "")
            if entry is None:
                break
            counts, total, types = entry
            p = (counts.get(ch, 0) + types * p) / (total + types)
        return p

    def bpc(self, s):
        return float(np.mean([-math.log2(self.prob(s[max(0, i - self.order + 1):i], s[i])) for i in range(1, len(s))]))


def _judge(train_text):
    global _JUDGE
    if _JUDGE is None or _JUDGE[0] != len(train_text):
        _JUDGE = len(train_text), _CharJudge(train_text)
    return _JUDGE[1]


def _char_classes(chars):
    """0 lowercase vowel, 1 other lowercase, 2 uppercase, 3 whitespace, 4 punctuation and the rest."""
    return torch.tensor([0 if c in "aeiou" else 1 if c.islower() else 2 if c.isupper() else 3 if c.isspace() else 4
                         for c in chars])


@torch.no_grad()
def _specialization(model, x, y, V, chars):
    """Measured on the frozen trained model: lesion one brain at a time and record the damage per character class.
    Each brain's profile is centred on its own mean damage, so a brain that is simply more important overall does not
    count as specialised; what remains is whether brains hurt *different* classes differently."""
    base = _eval_nll(model, x, y, V)
    klass = _char_classes(chars)[y.reshape(-1)]
    D = []
    for k in range(model.K):
        model.set_lesion(brains=[k])
        d = _eval_nll(model, x, y, V) - base
        D.append([d[klass == c].mean().item() for c in range(5)])
    model.set_lesion()
    D = np.array(D)
    shape = D - D.mean(1, keepdims=True)
    scale = np.abs(D).mean() + 1e-9
    return {"ablate_mean": float(D.mean()), "ablate_cv": float(D.mean(1).std() / scale),
            "special_index": float(shape.std(0).mean() / scale)}


@torch.no_grad()
def _generate(model, stoi, chars, V, n, temp=0.8):
    device, state, nxt, out = _device(model), None, torch.tensor([stoi["\n"]]), []
    for _ in range(n):
        logits, state = model(F.one_hot(nxt, V).float()[:, None, :].to(device), state=state, return_state=True)
        nxt = torch.multinomial(F.softmax(logits[0, -1] / temp, -1), 1).cpu()
        out.append(chars[int(nxt)])
    return "".join(out)


def text(model_fn, steps, sample_prob, lr=3e-3, T=64, B=32, gen_chars=None):
    """Character-level language modelling. Trains on the first 80% of the corpus and scores every character of the
    evaluated split (val while tuning, test for the reported run) in fixed windows."""
    gen_chars = gen_chars or GEN_CHARS
    raw, chars, stoi, data = _corpus()
    V = len(chars)
    (a, b), (c, d) = _split_bounds(len(data))
    train, held = data[a:b], data[c:d]
    model = model_fn(d_in=V, d_out=V)

    def batch():
        i = torch.randint(len(train) - T - 1, (B,))
        seq = train[i[:, None] + torch.arange(T + 1)[None, :]]
        return F.one_hot(seq[:, :-1], V).float(), seq[:, 1:]

    loss_fn = lambda o, y: F.cross_entropy(o.reshape(-1, V), y.reshape(-1))
    hx, hy = _windows(held, T)
    if EVAL_LIMIT:
        hx, hy = hx[:EVAL_LIMIT], hy[:EVAL_LIMIT]
    curve = {}
    every = max(steps // 8, 1)
    sec = fit(model, batch, loss_fn, steps, lr=lr, sample_prob=sample_prob, checkpoint_every=every,
              on_checkpoint=lambda s: curve.__setitem__(f"bpc_at_{s}", _eval_nll(model, hx[:128], hy[:128], V).mean().item()))
    res = {"bits_per_char": _eval_nll(model, hx, hy, V).mean().item(), "split": SPLIT, "sec_per_step": sec,
           "final_slope_per_1k": (curve[f"bpc_at_{steps}"] - curve[f"bpc_at_{steps - every}"]) / every * 1000, **curve}
    sample = _generate(model, stoi, chars, V, gen_chars)
    judge = _judge(raw[a:b])
    g8, g4 = _ngrams(sample, 8), _ngrams(sample, 4)
    train_text = raw[a:b]
    res.update(novel_8gram=float(np.mean([g not in train_text for g in g8])), distinct_4=len(set(g4)) / len(g4),
               judge_bpc=judge.bpc(sample), judge_bpc_real=judge.bpc(raw[c:c + gen_chars]),
               sample=sample.replace("\n", " / ")[:300])
    if getattr(model, "K", 1) > 1:
        res.update(_specialization(model, hx, hy, V, chars))
    res.update(model.diagnostics())
    return res


def _drift_mix(V, day, rng, strength):
    """Electrode-style drift: channels gradually get mixed with neighbours, rescaled, and some go dead."""
    mix = torch.eye(V)
    mix = (1 - strength * day) * mix + strength * day * torch.tensor(rng.permutation(np.eye(V)), dtype=torch.float32)
    gain = 1 + strength * day * torch.tensor(rng.normal(0, 1, V), dtype=torch.float32)
    dead = torch.tensor(rng.random(V) > min(0.06 * day, 0.4), dtype=torch.float32)
    return mix * (gain * dead)[None, :]


def drift(model_fn, steps, sample_prob, lr=3e-3, days=8, strength=0.09, B=64, local_rate=0.02, adapt_batches=20):
    """Learn on the job: train once, then face input statistics that shift every 'day' with no labels available.
    Compares staying frozen, plain Hebbian adaptation, and three-factor Hebbian gated by the model's own confidence."""
    Xtr, ytr, Xte, yte = _digits()
    model = model_fn(d_in=8, d_out=10)

    def batch():
        i = torch.randint(len(ytr), (B,))
        return _as_sequence(Xtr[i]), ytr[i]

    sec = fit(model, batch, lambda o, y: F.cross_entropy(o[:, -1], y), steps, lr=lr, sample_prob=sample_prob)
    rng = np.random.default_rng(0)
    mixes = [_drift_mix(8, d, rng, strength) for d in range(days + 1)]
    # frozen / three gradient-free local rules / two engineered label-free baselines (AdaBN, TENT)
    variants = {"frozen": model, "hebbian": copy.deepcopy(model), "gated": copy.deepcopy(model),
                "homeostatic": copy.deepcopy(model), "adabn": model, "tent": copy.deepcopy(model),
                "homeo_adabn": copy.deepcopy(model)}  # they fix different things: activity levels vs input statistics
    accs = {name: [] for name in variants}
    device = _device(model)
    clean = _as_sequence(Xtr[torch.randint(len(ytr), (512,))])
    mu0, sd0 = clean.mean((0, 1)), clean.std((0, 1))
    with torch.no_grad():  # day-0 activity levels the homeostatic copies will defend
        for name in ("homeostatic", "homeo_adabn"):
            variants[name](clean.to(device), record=True)
            variants[name].record_homeostasis_targets()
    tent = variants["tent"]
    for p in tent.parameters():
        p.requires_grad_(False)
    tent.homeo_gain.requires_grad_(True)  # TENT adapts the same per-neuron gains homeostasis controls
    tent_opt = torch.optim.Adam([tent.homeo_gain], lr=1e-2)
    for d, mix in enumerate(mixes):
        Xd = _as_sequence(Xte) @ mix
        Xa = _as_sequence(Xtr[torch.randint(len(ytr), (adapt_batches * B,))]) @ mix
        for name, m in variants.items():
            x_eval, x_adapt = Xd, Xa
            if name in ("adabn", "homeo_adabn"):
                # re-standardise each input channel of today's unlabelled data to day-0 statistics
                mu, sd = Xa.mean((0, 1)), Xa.std((0, 1)).clamp_min(1e-3)
                x_eval = (Xd - mu) / sd * sd0 + mu0
                x_adapt = (Xa - mu) / sd * sd0 + mu0
            elif d and name == "tent":
                for j in range(adapt_batches):  # gradient steps minimising prediction entropy, no labels
                    p = F.softmax(m(Xa[j * B:(j + 1) * B].to(device))[:, -1], -1)
                    loss = -(p * torch.log(p + 1e-8)).sum(1).mean()
                    tent_opt.zero_grad(set_to_none=True)
                    loss.backward()
                    tent_opt.step()
            if d and name in ("hebbian", "gated", "homeostatic", "homeo_adabn"):
                with torch.no_grad():  # unlabelled and gradient-free
                    for j in range(adapt_batches):
                        xb = x_adapt[j * B:(j + 1) * B].to(device)
                        out = m(xb, record=True)
                        if name in ("homeostatic", "homeo_adabn"):
                            m.homeostatic_update()
                            continue
                        mod = None
                        if name == "gated":
                            # third factor: own confidence, one value per recorded row (rows run time-first)
                            mod = F.softmax(out[:, -1], -1).max(1).values.repeat(xb.shape[1])
                        m.local_update(rate=local_rate, modulator=mod)
            with torch.no_grad():
                accs[name].append(_acc(F.softmax(m(x_eval.to(device))[:, -1], -1).cpu(), yte))
    res = {"sec_per_step": sec, "acc_day0": accs["frozen"][0]}
    for name, a in accs.items():
        res[f"acc_{name}_mean"] = float(np.mean(a[1:]))
        res[f"acc_{name}_final"] = a[-1]
    for name in ("hebbian", "gated", "homeostatic", "adabn", "tent", "homeo_adabn"):
        res[f"gain_{name}"] = res[f"acc_{name}_mean"] - res["acc_frozen_mean"]
    res.update(_diagnose(model, _as_sequence(Xte[:128])))
    return res


def _state_features(state, B):
    """Flatten a model's recurrent state to (B, features) for the hidden-cause probe."""
    if state.dim() == 3:
        return state.reshape(B, -1)
    return torch.cat([state[:B], state[B:]], 1) if state.shape[0] == 2 * B else state


CAUSAL = {"K": 6, "T": 30, "lapse": 0.1}  # world size / episode length / noise, overridable from run.py


def _causal_rollout(model, B, K, T, n_causes, lapse, eps, policy, invert=False):
    """One batch of episodes. A hidden 'cause' object turns the light on when pushed (with lapse noise); the model
    predicts the outcome of pushing each object and pushes whichever it is least sure about."""
    device = _device(model)
    cause_idx = torch.rand(B, K, device=device).argsort(1)[:, :n_causes]
    cause = torch.zeros(B, K, dtype=torch.bool, device=device).scatter_(1, cause_idx, True)
    light = torch.zeros(B, device=device)
    prev = torch.full((B,), -1, dtype=torch.long, device=device)
    state, loss, preds, actions = None, 0.0, [], []
    for t in range(T):
        onehot = torch.zeros(B, K, device=device)
        seen = prev >= 0
        if seen.any():
            onehot[seen] = F.one_hot(prev[seen], K).float()
        obs = torch.cat([onehot, light[:, None], torch.full((B, 1), t / T, device=device)], 1)
        out, state = model(obs[:, None, :], state=state, return_state=True)
        logits = out[:, 0, :K]
        with torch.no_grad():
            a = (-(torch.sigmoid(logits) - 0.5).abs()).argmax(1) if policy == "model" \
                else torch.randint(K, (B,), device=device)
            if eps:
                rand = torch.randint(K, (B,), device=device)
                a = torch.where(torch.rand(B, device=device) < eps, rand, a)
            p_on = torch.where(cause.gather(1, a[:, None]).squeeze(1), 1 - lapse, lapse)
            light = torch.bernoulli(1 - p_on if invert else p_on)
        loss = loss + F.binary_cross_entropy_with_logits(logits.gather(1, a[:, None]).squeeze(1), light)
        preds.append(logits.argmin(1) if invert else logits.argmax(1))
        actions.append(a)
        prev = a
    return loss / T, torch.stack(preds, 1), torch.stack(actions, 1), cause, state


def _causal_metrics(preds, actions, cause, T):
    ok = cause.gather(1, preds).float()
    settled = torch.flip(torch.cummin(torch.flip(ok, [1]), 1).values, [1])  # 1 where correct from here on
    identified = settled.max(1).values > 0
    id_step = torch.where(identified, settled.argmax(1).float(), torch.full_like(ok[:, 0], float(T)))
    pushes = torch.tensor([len(set(actions[i, :int(id_step[i])].tolist())) for i in range(len(actions))],
                          dtype=torch.float32)
    return {"frac_identified": identified.float().mean().item(), "id_step": id_step.mean().item(),
            "distinct_pushes": pushes.mean().item()}


def causal(model_fn, steps, sample_prob, lr=3e-3, K=None, T=None, B=64, lapse=None, eps=0.1, eval_eps=512):
    """Abduction test: infer which hidden object causes the light by choosing informative experiments."""
    K, T, lapse = K or CAUSAL["K"], T or CAUSAL["T"], CAUSAL["lapse"] if lapse is None else lapse
    model = model_fn(d_in=K + 2, d_out=K)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
    model.train()
    t0 = time.time()
    for _ in range(steps):
        loss = _causal_rollout(model, B, K, T, 1, lapse, eps, "model")[0]
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
    model.eval()
    res = {"sec_per_step": (time.time() - t0) / max(steps, 1)}
    with torch.no_grad():
        for policy in ("model", "random"):
            loss, preds, actions, cause, state = _causal_rollout(model, eval_eps, K, T, 1, lapse, 0.0, policy)
            tag = "" if policy == "model" else "_random"
            res[f"bce{tag}"] = loss.item()
            res.update({f"{k}{tag}": v for k, v in _causal_metrics(preds, actions, cause, T).items()})
            if policy == "model":
                feats, labels = _state_features(state, eval_eps).cpu().numpy(), cause.float().argmax(1).cpu().numpy()
        # rule-family transfer, never trained on: the cause now switches the light OFF instead of on
        _, preds, actions, cause_i, _ = _causal_rollout(model, eval_eps, K, T, 1, lapse, 0.0, "model", invert=True)
        res.update({f"inverted_{k}": v for k, v in _causal_metrics(preds, actions, cause_i, T).items()})
        # and a second rule: two objects work at once
        _, preds, actions, cause2, _ = _causal_rollout(model, eval_eps, K, T, 2, lapse, 0.0, "model")
        res.update({f"twocause_{k}": v for k, v in _causal_metrics(preds, actions, cause2, T).items()})
    half = eval_eps // 2
    probe = LogisticRegression(max_iter=1000).fit(feats[:half], labels[:half])
    res["probe_acc"] = float(probe.score(feats[half:], labels[half:]))
    res["probe_chance"] = 1.0 / K
    res.update(model.diagnostics())
    return res


NEURAL_DATA = None  # set by run.py: .npz of MICrONS movie frames + co-recorded neural traces


NEURAL_TRAIN_END = 0.7


def _neural_dataset(path, n_neurons=1000, downsample=2):
    """MICrONS two-photon window: stimulus frames and traces, keeping the most reliable neurons. Neuron selection and
    stimulus normalisation use the training part only, so nothing about the scored part leaks into them."""
    z = np.load(path)
    stim = torch.tensor(np.asarray(z["stim"], dtype=np.float32)).unsqueeze(1)
    stim = F.avg_pool2d(stim, downsample).flatten(1)
    n_tr = int(NEURAL_TRAIN_END * len(stim))
    stim = (stim - stim[:n_tr].mean()) / (stim[:n_tr].std() + 1e-6)
    resp = torch.nan_to_num(torch.tensor(np.asarray(z["traces"], dtype=np.float32)))
    # lag-1 autocorrelation: real calcium responses are smooth in time, noise is not
    keep = (resp[1:n_tr] * resp[:n_tr - 1]).mean(0).topk(min(n_neurons, resp.shape[1])).indices
    return stim, resp[:, keep]


def neural(model_fn, steps, sample_prob, lr=3e-3, T=40, B=32, n_neurons=1000):
    """Predict real MICrONS neural activity from the movie the mouse was watching. Trains on the first 70% of the
    recording and scores val (70-80%) or test (80-100%, the region v6 scored)."""
    stim, resp = _neural_dataset(NEURAL_DATA, n_neurons)
    n_train = int(NEURAL_TRAIN_END * len(stim))
    lo, hi = (n_train, int(0.8 * len(stim))) if SPLIT == "val" else (int(0.8 * len(stim)), len(stim))
    model = model_fn(d_in=stim.shape[1], d_out=resp.shape[1])

    def batch():
        i = torch.randint(0, n_train - T, (B,))
        idx = i[:, None] + torch.arange(T)[None, :]
        return stim[idx], resp[idx]

    sec = fit(model, batch, lambda o, y: F.mse_loss(o[:, 5:], y[:, 5:]), steps, lr=lr, sample_prob=sample_prob)
    starts = torch.arange(lo, hi - T, T)
    if EVAL_LIMIT:
        starts = starts[:EVAL_LIMIT]
    windows = starts[:, None] + torch.arange(T)[None, :]
    with torch.no_grad():
        pred = model(stim[windows].to(_device(model))).cpu()
    p = pred[:, 5:].reshape(-1, resp.shape[1])
    y = resp[windows][:, 5:].reshape(-1, resp.shape[1])
    p, y = p - p.mean(0), y - y.mean(0)
    r = (p * y).sum(0) / (p.norm(dim=0) * y.norm(dim=0) + 1e-8)
    res = {"sec_per_step": sec, "split": SPLIT, "mean_corr": r.mean().item(), "median_corr": r.median().item(),
           "frac_corr_gt0.2": (r > 0.2).float().mean().item(),
           "top100_corr": r.topk(min(100, len(r))).values.mean().item(), "n_neurons": float(len(r))}
    res.update(_diagnose(model, stim[windows[:8]]))
    return res


TRAIN_SPEEDS = torch.tensor([0.15, 0.3, 0.45, 0.6])
NOVEL_SPEEDS = torch.tensor([0.225, 0.375, 0.525])


def _oscillators(bs, T, speeds, noise=0.05):
    """Two 2D rotations whose angular speeds jump to new values at a random time: a small world whose rules change."""
    sw = torch.randint(25, 36, (bs,))
    i1 = torch.randint(len(speeds), (bs, 2))
    i2 = (i1 + torch.randint(1, len(speeds), (bs, 2))) % len(speeds)
    w1, w2 = speeds[i1][:, None, :], speeds[i2][:, None, :]
    t = torch.arange(T, dtype=torch.float32)[None, :, None]
    s = sw[:, None, None].float()
    phi = torch.rand(bs, 1, 2) * 2 * math.pi + torch.where(t < s, w1 * t, w1 * s + w2 * (t - s))
    clean = torch.cat([torch.cos(phi), torch.sin(phi)], -1)
    return clean + noise * torch.randn_like(clean), clean, sw


def _phase_errors(pred, clean, sw):
    """NRMSE of next-frame predictions before the rule change, in the 5 frames after it, and 10-19 frames after."""
    err = ((pred[:, :-1] - clean[:, 1:]) ** 2).mean(-1)  # err[:, t] is the error predicting frame t+1
    t = torch.arange(err.shape[1])[None, :]
    s = sw[:, None]
    windows = {"pre": (t >= 10) & (t < s - 1), "early": (t >= s - 1) & (t < s + 4), "late": (t >= s + 9) & (t < s + 19)}
    return {k: math.sqrt((err * m).sum().item() / m.sum().item() / 0.5) for k, m in windows.items()}


def worldmodel(model_fn, steps, sample_prob, lr=3e-3, T=60, B=32):
    model = model_fn(d_in=4, d_out=4)

    def batch():
        obs, clean, _ = _oscillators(B, T, TRAIN_SPEEDS)
        return obs, clean

    sec = fit(model, batch, lambda o, y: F.mse_loss(o[:, 5:-1], y[:, 6:]), steps, lr=lr, sample_prob=sample_prob)
    res = {"sec_per_step": sec}
    for name, speeds in (("seen", TRAIN_SPEEDS), ("novel", NOVEL_SPEEDS)):
        obs, clean, sw = _oscillators(512, T, speeds)
        with torch.no_grad():
            pred = model(obs.to(_device(model))).cpu()
        errs, persist = _phase_errors(pred, clean, sw), _phase_errors(obs, clean, sw)
        for k, v in errs.items():
            res[f"wm_{name}_{k}"] = v
        res[f"wm_{name}_recovery"] = errs["late"] / errs["pre"]
        res[f"wm_{name}_late_vs_persist"] = errs["late"] / persist["late"]
    res.update(_diagnose(model, obs[:64]))
    return res



STREAM = {"chunk": 32, "n_chunks": 144, "B": 4, "train_B": 32, "delta_rates": (0.02,), "dyneval_lrs": (3e-4,),
          "fast_rate": 0.04, "fast_decay": 0.7, "fast_clamp": 0.1}
# v12's bolt-on Hebbian fast weights, now clamped to +-10% per synapse; kept as controls, not as the method.
# surprise writes when the prediction failed, confirm when it held - v12 reversed the synthetic-data ordering, so both run
HEBB_RULES = {"hebb_safe": 0.0, "surprise_safe": 1.0, "confirm_safe": -1.0}


def _reset_memory(state):
    """Wipe the fast memory but keep the network's own activity: isolates what the memory carries across chunks."""
    if isinstance(state, tuple):
        x, S, r = state
        return x, torch.zeros_like(S), torch.zeros_like(r)
    return state


def stream(model_fn, steps, sample_prob, lr=3e-3, T=64):
    """Learning on the job. Pretrain statefully on the first 80% (every training stream reads consecutive windows and
    carries its state, so a memory can learn to be useful beyond one window), then read the evaluated split as B long
    streams. Every chunk is scored *before* anything is learned from it. What changes, per variant:
      frozen    nothing but the recurrent state - and, for :fm models, the trained fast memory, whose delta-rule writes
                are part of the forward pass: learning from the stream with no gradient at all
      memreset  :fm models only - the memory is wiped at every chunk, isolating what it carries forward
      delta     local delta rule on the readout (algebraically one SGD step on that layer alone)
      dyneval   dynamic evaluation (Krause et al. 2018): an Adam step on every weight after every chunk - the
                backprop ceiling the gradient-free variants are measured against
      *_safe    the bolt-on Hebbian rules, stability-clamped, as controls (connectome models without memory only)
    With several delta_rates / dyneval_lrs (hyperparameter tuning) variants are named delta@rate, dyneval@lr."""
    cfg = STREAM
    chunk, n_chunks, B, train_B = cfg["chunk"], cfg["n_chunks"], cfg["B"], cfg["train_B"]
    raw, chars, stoi, data = _corpus()
    V = len(chars)
    (a, b), (c, d) = _split_bounds(len(data))
    train, held = data[a:b], data[c:d]
    model = model_fn(d_in=V, d_out=V)
    device = _device(model)
    pos = torch.randint(len(train) - T - 1, (train_B,))

    def batch():  # consecutive windows per stream, for stateful training
        idx = (pos[:, None] + torch.arange(T + 1)[None, :]) % len(train)
        pos.add_(T).remainder_(len(train) - T - 1)
        seq = train[idx]
        return F.one_hot(seq[:, :-1], V).float(), seq[:, 1:]

    loss_fn = lambda o, y: F.cross_entropy(o.reshape(-1, V), y.reshape(-1))
    sec = fit(model, batch, loss_fn, steps, lr=lr, sample_prob=sample_prob, stateful=True)
    hx, hy = _windows(held, T)
    if EVAL_LIMIT:
        hx, hy = hx[:EVAL_LIMIT], hy[:EVAL_LIMIT]
    res = {"sec_per_step": sec, "split": SPLIT, "bpc_windows": _eval_nll(model, hx, hy, V).mean().item()}
    for p in model.parameters():
        p.requires_grad_(False)

    need = chunk * n_chunks + 1
    stride = (len(held) - need) // B
    assert stride >= need, "streams would overlap"
    seq = held[torch.arange(B)[:, None] * stride + torch.arange(need)[None, :]]

    tag = lambda name, values, v: name if len(values) == 1 else f"{name}@{v:g}"
    variants = {"frozen": ("frozen", None)}
    if getattr(model, "memory", None) is not None:
        variants["memreset"] = ("memreset", None)
    variants.update({tag("delta", cfg["delta_rates"], r): ("delta", r) for r in cfg["delta_rates"]})
    variants.update({tag("dyneval", cfg["dyneval_lrs"], l): ("dyneval", l) for l in cfg["dyneval_lrs"]})
    if hasattr(model, "pre") and getattr(model, "memory", None) is None:
        variants.update({name: (name, None) for name in HEBB_RULES})

    def run(kind, rate):
        m = copy.deepcopy(model)
        opt = None
        if kind == "dyneval":
            for p in m.parameters():
                p.requires_grad_(True)
            opt = torch.optim.Adam(m.parameters(), lr=rate)
        hebb = HEBB_RULES.get(kind)
        if hebb is not None:
            m.enable_fast(True, rate=cfg["fast_rate"], decay=cfg["fast_decay"], clamp=cfg["fast_clamp"])
        bpcs, state = [], None
        for ci in range(n_chunks):
            x = seq[:, ci * chunk:(ci + 1) * chunk]
            y = seq[:, ci * chunk + 1:(ci + 1) * chunk + 1].to(device)
            if kind == "memreset":
                state = _reset_memory(state)
            with torch.set_grad_enabled(opt is not None):
                logits, new_state = m(F.one_hot(x, V).float().to(device), state=state,
                                      record=hebb is not None or kind == "delta", return_state=True)
                nll = F.cross_entropy(logits.reshape(-1, V), y.reshape(-1), reduction="none").reshape(y.shape)
            bpcs.append(nll.mean().item() / math.log(2))  # scored before this chunk teaches anything
            if opt is not None:
                opt.zero_grad(set_to_none=True)
                nll.mean().backward()
                opt.step()
            with torch.no_grad():  # rows run time-first in everything the model records
                if hebb is not None:
                    m.fast_write(hebb * nll.T.reshape(-1) if hebb else None)
                if kind == "delta":
                    m.delta_readout_update(F.softmax(logits, -1).transpose(0, 1).reshape(-1, V),
                                           y.T.reshape(-1), rate=rate)
            state = detach_state(new_state)
        return bpcs

    q = max(n_chunks // 4, 1)
    for name, (kind, rate) in variants.items():
        bpcs = run(kind, rate)
        res[f"bpc_{name}"] = float(np.mean(bpcs))
        res[f"late_{name}"] = float(np.mean(bpcs[-q:]))
        res[f"within_{name}"] = float(np.mean(bpcs[:q]) - np.mean(bpcs[-q:]))
        if name != "frozen":
            res[f"gain_{name}"] = res["bpc_frozen"] - res[f"bpc_{name}"]
    if "memreset" in variants:  # > 0: carrying the memory's own writes forward helped, with no gradient anywhere
        res["memory_carry_gain"] = res["bpc_memreset"] - res["bpc_frozen"]
    res.update(model.diagnostics())
    return res


TASKS = {"memory": memory, "narma10": narma10, "digits": digits, "decision": decision, "continual": continual,
         "worldmodel": worldmodel, "neural": neural, "causal": causal, "text": text, "drift": drift,
         "stream": stream}

In [ ]:
%%writefile /kaggle/working/multiverse/run.py
import argparse
import json
import os
import time

import numpy as np
import torch

import tasks
from connectome import load_microns, load_npz, synthetic
from model import MODES, PHASE_MODES, REWIRE_RULES, GRUBrain, Multiverse, PhaseBrain, TransformerLM
from tasks import TASKS

HERE = os.path.dirname(os.path.abspath(__file__))
DEFAULT_DATA = os.path.join(HERE, "data", "microns_v1181_axon.npz")


def build_connectome(args):
    if args.connectome == "synthetic":
        real = synthetic(n=args.n_neurons, seed=0)
    elif args.data.endswith(".npz"):
        real = load_npz(args.data)
    else:
        real = load_microns(args.data, args.proofread)
    conn = {"real": real, "shuffled": real.shuffled(seed=1), "random": real.randomized(seed=1)}[args.wiring]
    return real, conn


def parse_spec(spec):
    """A condition is 'mode[+rule][@readout][:option...]'.
      mode      single, separate, quantum, ... or a baseline: gru[-hidden], transformer[-d_model-layers]
      +rule     rewiring rule; the prefix 'brain_' makes it per-brain (each brain prunes and grows its own synapses)
      @readout  l5 (layer-5 output cells), excitatory, all
      options   fm (gated delta-rule fast memory), lr=, wd=, do= (dropout), churn= (fraction rewired per event)
    e.g. 'separate+brain_rigl@excitatory:churn=0.02', 'gru-200:do=0.2:wd=0.1:lr=0.003', 'single@excitatory:fm'.
    The hyperparameters live in the condition string, so every result says exactly how it was produced."""
    head, *opts = spec.split(":")
    options = {}
    for opt in opts:
        key, _, value = opt.partition("=")
        options[key] = float(value) if value else True
    body, _, readout = head.partition("@")
    mode, _, rule = body.partition("+")
    return mode, rule or "none", readout, options


def make_model_fn(spec, conn, args, device, local_sigma):
    mode, rule, readout, o = parse_spec(spec)
    dropout, memory = o.get("do", 0.0), bool(o.get("fm", False))
    base, *size = mode.split("-")
    if base == "gru":
        hidden = int(size[0]) if size else args.gru_hidden
        return lambda d_in, d_out: GRUBrain(d_in, d_out, hidden=hidden, dropout=dropout, memory=memory).to(device)
    if base == "transformer":
        assert not memory, "the transformer has attention; no separate fast memory"
        d_model, n_layer = (int(size[0]), int(size[1])) if size else (64, 3)
        return lambda d_in, d_out: TransformerLM(d_in, d_out, d_model, n_layer, dropout=dropout).to(device)
    per_brain = rule.startswith("brain_")
    rule = rule[len("brain_"):] if per_brain else rule
    assert mode in MODES + PHASE_MODES and rule in REWIRE_RULES, spec
    out_idx = {"l5": conn.output_idx, "excitatory": np.flatnonzero(~conn.inhibitory),
               "all": np.arange(conn.n)}[readout or args.readout]
    common = dict(rewire=rule, rewire_every=args.rewire_every, rewire_frac=o.get("churn", args.rewire_frac),
                  local_sigma=local_sigma)
    if mode in PHASE_MODES:
        assert not per_brain and not memory, spec
        return lambda d_in, d_out: PhaseBrain(conn, d_in, d_out, conn.input_idx, out_idx, mode, **common).to(device)
    return lambda d_in, d_out: Multiverse(conn, d_in, d_out, conn.input_idx, out_idx, args.K, mode,
                                          plastic=args.plastic, per_brain_rewire=per_brain,
                                          grow_slack=args.grow_slack, dropout=dropout, memory=memory,
                                          **common).to(device)


def print_summary(results):
    print("\n=== summary (mean over seeds) ===")
    for task, by_mode in results.items():
        keys = [k for k, v0 in next(iter(by_mode.values()))[0].items()
                if isinstance(v0, (int, float)) and all(k in r for runs in by_mode.values() for r in runs)]
        print(f"\n== {task} ==")
        print("condition".ljust(28) + "".join(k[:18].rjust(20) for k in keys))
        for mode, runs in by_mode.items():
            print(mode.ljust(28) + "".join(f"{np.mean([r[k] for r in runs]):20.4g}" for k in keys))


def floats(text):
    return tuple(float(v) for v in text.split(",") if v)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--connectome", choices=["microns", "synthetic"], default="microns")
    ap.add_argument("--data", default=DEFAULT_DATA, help=".npz subset, or the full MICrONS .h5")
    ap.add_argument("--proofread", choices=["axon", "dendrite"], default="axon", help=".h5 only")
    ap.add_argument("--n-neurons", type=int, default=400, help="synthetic only")
    ap.add_argument("--wiring", choices=["real", "shuffled", "random"], default="real")
    ap.add_argument("--neural-data", default="", help=".npz of MICrONS stimulus frames + neural traces")
    ap.add_argument("--modes", default=",".join(MODES), help="comma-separated conditions, see parse_spec")
    ap.add_argument("--tasks", default=",".join(TASKS))
    ap.add_argument("--split", choices=["val", "test"], default="test",
                    help="val while tuning hyperparameters; test only for the reported run")
    ap.add_argument("--K", type=int, default=6)
    ap.add_argument("--steps", type=int, default=300)
    ap.add_argument("--task-steps", default="", help="per-task overrides, e.g. text=4000,neural=2000")
    ap.add_argument("--lr", type=float, default=3e-3, help="default when a condition has no lr= option")
    ap.add_argument("--weight-decay", type=float, default=0.0, help="default when a condition has no wd= option")
    ap.add_argument("--sample-prob", type=float, default=0.5)
    ap.add_argument("--seeds", type=int, default=1)
    ap.add_argument("--seed-offset", type=int, default=0, help="first seed, so replications can use fresh seeds")
    ap.add_argument("--plastic", action="store_true", help="learn a strength for every synapse (signs fixed)")
    ap.add_argument("--rewire-every", type=int, default=100)
    ap.add_argument("--rewire-frac", type=float, default=0.1, help="default when a condition has no churn= option")
    ap.add_argument("--gru-hidden", type=int, default=512)
    ap.add_argument("--readout", choices=["l5", "excitatory", "all"], default="l5",
                    help="default readout population when a condition has no @readout suffix")
    ap.add_argument("--grow-slack", type=float, default=0.15,
                    help="spare synapse slots per brain, as a fraction of the connectome, for free genesis")
    ap.add_argument("--delta-rates", default="0.02", help="stream: readout delta-rule rate(s), comma-separated")
    ap.add_argument("--dyneval-lrs", default="3e-4", help="stream: dynamic-evaluation Adam lr(s), comma-separated")
    ap.add_argument("--fast-rate", type=float, default=0.04, help="stream: Hebbian-control write rate")
    ap.add_argument("--fast-decay", type=float, default=0.7, help="stream: Hebbian-control decay")
    ap.add_argument("--fast-clamp", type=float, default=0.1, help="stream: max fractional change of any synapse")
    ap.add_argument("--stream-chunks", type=int, default=144, help="stream: chunks of text read after freezing")
    ap.add_argument("--stream-b", type=int, default=4, help="stream: parallel streams (few and long beats many)")
    ap.add_argument("--causal-k", type=int, default=6, help="objects in the hidden-cause world")
    ap.add_argument("--causal-t", type=int, default=30, help="steps per abduction episode")
    ap.add_argument("--causal-lapse", type=float, default=0.1)
    ap.add_argument("--deadline", type=float, default=0.0,
                    help="unix time after which no new run starts (a Kaggle session must end to save anything)")
    ap.add_argument("--out-dir", default=os.path.join(HERE, "results"))
    ap.add_argument("--threads", type=int, default=0)
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    args = ap.parse_args()

    if args.threads:
        torch.set_num_threads(args.threads)
    tasks.SPLIT = args.split
    tasks.NEURAL_DATA = args.neural_data
    tasks.CAUSAL = {"K": args.causal_k, "T": args.causal_t, "lapse": args.causal_lapse}
    tasks.STREAM = {**tasks.STREAM, "n_chunks": args.stream_chunks, "B": args.stream_b,
                    "delta_rates": floats(args.delta_rates), "dyneval_lrs": floats(args.dyneval_lrs),
                    "fast_rate": args.fast_rate, "fast_decay": args.fast_decay, "fast_clamp": args.fast_clamp}
    task_steps = {k: int(v) for k, v in (item.split("=") for item in args.task_steps.split(",") if item)}
    device = torch.device(args.device)
    real, conn = build_connectome(args)
    # local growth uses the real connectome's typical connection length, whatever wiring we start from
    local_sigma = float(np.median(np.linalg.norm(real.xyz[real.pre] - real.xyz[real.post], axis=1))) / 1000
    print(f"{conn.summary()} | device={device} | split={args.split} | local growth sigma {local_sigma:.3f}", flush=True)

    os.makedirs(args.out_dir, exist_ok=True)
    # pid keeps parallel jobs that start in the same second from overwriting each other's results
    tag = f"{conn.name}{'-plastic' if args.plastic else ''}_{args.split}_{time.strftime('%Y%m%d-%H%M%S')}_{os.getpid()}"
    path = os.path.join(args.out_dir, f"{tag}.json")

    results, last_wall = {}, {}
    # seed-first ordering keeps partial results balanced across conditions if a session is cut short
    for seed in range(args.seed_offset, args.seed_offset + args.seeds):
        for task in args.tasks.split(","):
            steps = task_steps.get(task, args.steps)
            for spec in args.modes.split(","):
                expected = last_wall.get((task, spec), max(last_wall.values(), default=0.0))
                if args.deadline and time.time() + expected > args.deadline:
                    print(f"deadline: not starting [{task}] {spec} seed{seed} (expected {expected / 60:.0f} min)",
                          flush=True)
                    continue
                torch.manual_seed(seed)
                np.random.seed(seed)
                _, _, _, o = parse_spec(spec)
                tasks.WEIGHT_DECAY = o.get("wd", args.weight_decay)
                lr = o.get("lr", args.lr)
                t0 = time.time()
                res = TASKS[task](make_model_fn(spec, conn, args, device, local_sigma), steps, args.sample_prob, lr=lr)
                res.update(steps=steps, wall_sec=time.time() - t0, seed=seed, condition=spec, lr=lr,
                           weight_decay=tasks.WEIGHT_DECAY)
                last_wall[(task, spec)] = res["wall_sec"]
                results.setdefault(task, {}).setdefault(spec, []).append(res)
                shown = ", ".join(f"{k}={v:.3g}" if isinstance(v, (int, float)) else f"{k}={str(v)[:120]!r}"
                                  for k, v in res.items())
                print(f"[{task}] {spec} seed{seed}: {shown}", flush=True)
                with open(path, "w") as f:
                    json.dump({"args": vars(args), "connectome": conn.summary(), "results": results}, f, indent=1)

    if results:
        print_summary(results)
    print(f"\nsaved {path}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/multiverse/tune.py
"""Hyperparameter tuning for v13, run locally on CPU on the *validation* split, seed 0. Picks are written to
tuned_v13.json and become the condition strings of the reported (test-split) Kaggle run.

  python tune.py grid     # stage 1: text and neural grids
  python tune.py stream   # stage 2: delta-rule rate and dynamic-evaluation lr, using the stage-1 core picks
  python tune.py select   # choose the best config per family and write tuned_v13.json

Every family gets the same kind of search (learning rate x weight decay, plus dropout where the model overfits),
so no baseline is left untuned while the brain is favoured - the flaw in v1-v12."""
import glob
import itertools
import json
import os
import subprocess
import sys
import time

HERE = os.path.dirname(os.path.abspath(__file__))
OUT = os.path.join(HERE, "results", "tuning_v13")
PICKS = os.path.join(HERE, "tuned_v13.json")  # pipeline.py redirects both
NEURAL = os.environ.get("NEURAL_DATA", r"E:\microns_data\functional_scan.npz")
# the brain's sparse CPU matmul barely uses a second thread, so dense GRU/transformer jobs can share the machine
WORKERS, THREADS = int(os.environ.get("WORKERS", 4)), int(os.environ.get("THREADS", 2))

GRID = {  # family: (task, steps, lrs, dropouts, weight decays)
    "text/single@excitatory": ("text", 4000, (3e-3, 1e-2), (0.0,), (0.0, 0.1)),
    "text/gru-200": ("text", 4000, (3e-3, 1e-2), (0.0, 0.2), (0.0, 0.1)),
    "text/gru-512": ("text", 4000, (3e-3, 1e-2), (0.0, 0.2), (0.0, 0.1)),
    "text/transformer-64-3": ("text", 4000, (1e-3, 3e-3), (0.0, 0.1), (0.0, 0.1)),
    "text/transformer-128-4": ("text", 4000, (1e-3, 3e-3), (0.0, 0.1), (0.0, 0.1)),
    "neural/single": ("neural", 2000, (3e-3, 1e-2), (0.0,), (0.0, 0.1)),
    "neural/gru-128": ("neural", 2000, (3e-3, 1e-2), (0.0, 0.2), (0.0, 0.1)),
    "neural/gru-512": ("neural", 2000, (3e-3, 1e-2), (0.0, 0.2), (0.0, 0.1)),
}
# the brain does not overfit (v12 final slope -0.013), so it gets no dropout axis; every GRU and transformer does
COST = {"single": 44, "gru-512": 8, "transformer-128-4": 14, "transformer-64-3": 5, "gru-200": 3, "gru-128": 2}
STREAM_CORES = ("single@excitatory", "gru-200")
DELTA_RATES, DYNEVAL_LRS = "0.005,0.02,0.05", "1e-4,3e-4,1e-3"


def spec(mode, lr, do, wd):
    return f"{mode}:lr={lr:g}" + (f":do={do:g}" if do else "") + (f":wd={wd:g}" if wd else "")


def grid_jobs():
    jobs = []
    for family, (task, steps, lrs, dos, wds) in GRID.items():
        mode = family.split("/", 1)[1]
        cost = COST[mode.split("@")[0]] * (0.3 if task == "neural" else 1.0)
        for lr, do, wd in itertools.product(lrs, dos, wds):
            jobs.append((cost, task, steps, spec(mode, lr, do, wd), family))
    return sorted(jobs, reverse=True)  # longest first keeps the pool busy to the end


def command(task, steps, modes, extra=()):
    cmd = [sys.executable, "-u", os.path.join(HERE, "run.py"), "--plastic", "--split", "val", "--seeds", "1",
           "--device", "cpu", "--threads", str(THREADS), "--out-dir", OUT, "--tasks", task,
           "--task-steps", f"{task}={steps}", "--modes", modes, *extra]
    return cmd + (["--neural-data", NEURAL] if task == "neural" else [])


def run_pool(jobs):
    os.makedirs(os.path.join(OUT, "logs"), exist_ok=True)
    env = {**os.environ, "OMP_NUM_THREADS": str(THREADS), "MKL_NUM_THREADS": str(THREADS)}
    pending, running, t0 = list(jobs), [], time.time()
    while pending or running:
        while pending and len(running) < WORKERS:
            name, cmd = pending.pop(0)
            log = open(os.path.join(OUT, "logs", name.replace("/", "_").replace(":", "_") + ".log"), "w")
            running.append((name, subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env), log))
        time.sleep(5)
        for item in [r for r in running if r[1].poll() is not None]:
            running.remove(item)
            item[2].close()
            print(f"{(time.time() - t0) / 60:6.1f} min  done ({item[1].returncode})  {item[0]}  "
                  f"[{len(pending)} queued, {len(running)} running]", flush=True)


def load_runs():
    runs = []
    for f in glob.glob(os.path.join(OUT, "*.json")):
        for task, conds in json.load(open(f))["results"].items():
            for cond, rs in conds.items():
                runs += [(task, cond, r) for r in rs]
    return runs


def select():
    """Best validation score per family: lowest bits/char for text, highest mean correlation for neural."""
    picks, table = {}, []
    for family, (task, *_rest) in GRID.items():
        mode = family.split("/", 1)[1]
        cands = [(c, r) for t, c, r in load_runs() if t == task and c.split(":")[0] == mode]
        key = (lambda cr: cr[1]["bits_per_char"]) if task == "text" else (lambda cr: -cr[1]["mean_corr"])
        cands.sort(key=key)
        if cands:
            picks[family] = cands[0][0]
            table += [(family, c, r.get("bits_per_char", r.get("mean_corr"))) for c, r in cands]
    for family, cond, score in table:
        print(f"{'*' if picks.get(family) == cond else ' '} {family:26s} {cond:45s} {score:.4f}")
    stream = [r for t, c, r in load_runs() if t == "stream"]
    if stream:
        for key, prefix in (("delta", "gain_delta@"), ("dyneval", "gain_dyneval@")):
            rates = sorted({k[len(prefix):] for r in stream for k in r if k.startswith(prefix)})
            mean_gain = {v: sum(r[prefix + v] for r in stream) / len(stream) for v in rates}
            picks[f"stream/{key}"] = max(mean_gain, key=mean_gain.get)
            print(f"  stream {key}: mean gain by rate {mean_gain} -> {picks[f'stream/{key}']}")
    json.dump(picks, open(PICKS, "w"), indent=1)
    print(f"wrote {PICKS}")
    return picks


def done_in_family(family):
    task, mode = family.split("/", 1)
    return sum(1 for t, c, _ in load_runs() if t == task and c.split(":")[0] == mode)


if __name__ == "__main__":
    stage = sys.argv[1]
    if stage == "grid":
        run_pool([(f"{fam}/{sp}", command(task, steps, sp)) for _, task, steps, sp, fam in grid_jobs()])
    elif stage == "grid-rest":  # every grid job outside the families named after it (already running elsewhere)
        skip = set(sys.argv[2:])
        first = {f"text/{core}" for core in STREAM_CORES}  # the stream stage waits on these picks
        jobs = sorted((j for j in grid_jobs() if j[4] not in skip), key=lambda j: (j[4] not in first, -j[0]))
        run_pool([(f"{fam}/{sp}", command(task, steps, sp)) for _, task, steps, sp, fam in jobs])
    elif stage == "wait":  # wait <family> <n>: block until n results of that family exist
        family, n = sys.argv[2], int(sys.argv[3])
        while done_in_family(family) < n:
            time.sleep(30)
        print(f"{family}: {n} results present", flush=True)
    elif stage == "stream":
        picks = json.load(open(os.path.join(HERE, "tuned_v13.json")))
        jobs = [(f"stream/{core}", command("stream", 4000, picks[f"text/{core}"],
                                           ["--delta-rates", DELTA_RATES, "--dyneval-lrs", DYNEVAL_LRS]))
                for core in STREAM_CORES]
        run_pool(jobs)
    elif stage == "select":
        select()

In [ ]:
%%writefile /kaggle/working/multiverse/pipeline.py
"""v13 end to end in one Kaggle session: tune on the validation split, pick hyperparameters, then run the
pre-registered test-split conditions (PREREGISTRATION.md).

Every step is a job that names the tuned values it needs. A GPU takes the highest-priority job whose inputs exist,
so a question starts the moment its hyperparameters are picked. One process per GPU (never packed); primary
questions come before exploratory ones; reported runs are split per seed so the GPUs balance and a session that
nears the deadline loses whole exploratory seeds, not primaries.

  python pipeline.py --deadline <unix time> --neural <functional_scan.npz> [--gpus 2] [--smoke]"""
import argparse
import json
import os
import subprocess
import sys
import time

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
import tune  # noqa: E402


def opts(picks, family):
    """The tuned options of a family's pick, e.g. 'lr=0.01:wd=0.1'."""
    return picks[family].split(":", 1)[1] if ":" in picks[family] else ""


def with_opts(head, o):
    return f"{head}:{o}" if o else head


def build_jobs(seeds, steps):
    """(name, needs, provides, argv builder). The argv builder receives the current picks."""
    text = lambda n: ["--tasks", "text", "--task-steps", f"text={n}"]
    jobs = []
    for family, (task, fsteps, lrs, dos, wds) in tune.GRID.items():
        mode = family.split("/", 1)[1]
        specs = [tune.spec(mode, lr, do, wd) for lr in lrs for do in dos for wd in wds]
        jobs.append((f"tune {family}", [], [family], "val", lambda p, t=task, s=specs: [
            "--tasks", t, "--task-steps", f"{t}={steps[t]}", "--modes", ",".join(s), "--seeds", "1"]))
    jobs.append(("tune stream rates", ["text/single@excitatory", "text/gru-200"], ["stream/delta", "stream/dyneval"],
                 "val", lambda p: ["--tasks", "stream", "--task-steps", f"stream={steps['stream']}", "--seeds", "1",
                                   "--delta-rates", tune.DELTA_RATES, "--dyneval-lrs", tune.DYNEVAL_LRS, "--modes",
                                   f"{p['text/single@excitatory']},{p['text/gru-200']}"]))
    brain = lambda p: opts(p, "text/single@excitatory")
    main = {  # question: (needs, argv builder) - one job per seed
        "Q4 neural": (["neural/single", "neural/gru-128", "neural/gru-512"], lambda p: [
            "--tasks", "neural", "--task-steps", f"neural={steps['neural']}", "--modes",
            ",".join([with_opts("single", opts(p, "neural/single")), p["neural/gru-128"], p["neural/gru-512"]])]),
        "Q1 baselines": (["text/single@excitatory", "text/gru-200", "text/transformer-64-3"], lambda p: [
            *text(steps["text"]), "--modes",
            ",".join([p["text/single@excitatory"], p["text/gru-200"], p["text/transformer-64-3"]])]),
        "Q2 stream": (["text/single@excitatory", "text/gru-200", "stream/delta", "stream/dyneval"], lambda p: [
            "--tasks", "stream", "--task-steps", f"stream={steps['stream']}", "--delta-rates", p["stream/delta"],
            "--dyneval-lrs", p["stream/dyneval"], "--modes", ",".join([
                p["text/single@excitatory"], with_opts("single@excitatory:fm", brain(p)),
                p["text/gru-200"], with_opts("gru-200:fm", opts(p, "text/gru-200"))])]),
        "Q3 clones": (["text/single@excitatory"], lambda p: [
            *text(steps["text"]), "--modes", with_opts("separate@excitatory", brain(p))]),
        "Q3 rigl": (["text/single@excitatory"], lambda p: [
            *text(steps["text"]), "--modes", with_opts("separate+brain_rigl@excitatory:churn=0.02", brain(p))]),
    }
    exploratory = {
        "Q1 larger": (["text/gru-512", "text/transformer-128-4"], lambda p: [
            *text(steps["text"]), "--modes", f"{p['text/gru-512']},{p['text/transformer-128-4']}"]),
        "Q3 heavy churn": (["text/single@excitatory"], lambda p: [
            *text(steps["text"]), "--modes", with_opts("separate+brain_rigl@excitatory:churn=0.1", brain(p))]),
    }
    for group in (main, exploratory):  # seed-major: every question's seed 0 before any seed 1
        for s in range(seeds):
            for q, (needs, argv) in group.items():
                jobs.append((f"{q} seed{s}", needs, [], "test",
                             lambda p, a=argv, s=s: a(p) + ["--seeds", "1", "--seed-offset", str(s)]))
    return jobs


def order(jobs):
    """Tuning the families that gate primaries first, then primaries seed-major, then the rest."""
    gate = {"tune text/single@excitatory": 0, "tune text/gru-200": 1, "tune text/transformer-64-3": 2,
            "tune neural/single": 3, "tune neural/gru-128": 4, "tune neural/gru-512": 5, "tune stream rates": 6}
    late = ("tune text/gru-512", "tune text/transformer-128-4")
    def key(item):
        i, (name, *_rest) = item
        if name in gate:
            return (0, gate[name])
        if name in late:
            return (2, i)
        return (1 if not name.startswith(("Q1 larger", "Q3 heavy")) else 3, i)
    return [j for _, j in sorted(enumerate(jobs), key=key)]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--deadline", type=float, required=True)
    ap.add_argument("--neural", required=True)
    ap.add_argument("--gpus", type=int, default=2)
    ap.add_argument("--seeds", type=int, default=5)
    ap.add_argument("--out", default=os.path.join(HERE, "results"))
    ap.add_argument("--smoke", action="store_true", help="tiny steps on CPU, to test the pipeline itself")
    args = ap.parse_args()

    steps = {"text": 4000, "neural": 2000, "stream": 4000}
    extra = []
    if args.smoke:
        steps = {"text": 6, "neural": 4, "stream": 6}
        extra = ["--stream-chunks", "2", "--stream-b", "2", "--rewire-every", "2", "--device", "cpu", "--threads", "1"]
    tune.OUT = os.path.join(args.out, "tuning_v13")
    tune.PICKS = os.path.join(args.out, "tuned_v13.json")
    test_out = os.path.join(args.out, "v13")
    logs = os.path.join(args.out, "logs")
    for d in (tune.OUT, test_out, logs):
        os.makedirs(d, exist_ok=True)

    pending = order(build_jobs(args.seeds, steps))
    picks, provided, failed, running, t0 = {}, set(), set(), {}, time.time()
    free = list(range(args.gpus))
    while pending or running:
        for gpu in list(free):
            ready = next((j for j in pending if all(n in provided for n in j[1])), None)
            if ready is None or time.time() > args.deadline:
                break
            pending.remove(ready)
            name, needs, provides, split, argv = ready
            out_dir = tune.OUT if split == "val" else test_out
            cmd = [sys.executable, "-u", os.path.join(HERE, "run.py"), "--plastic", "--sample-prob", "0",
                   "--split", split, "--deadline", str(args.deadline), "--out-dir", out_dir, *extra, *argv(picks)]
            if "neural" in cmd[cmd.index("--tasks") + 1]:
                cmd += ["--neural-data", args.neural]
            log = open(os.path.join(logs, name.replace(" ", "_").replace("/", "-") + ".log"), "w")
            env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
            running[gpu] = (ready, subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env), log, time.time())
            free.remove(gpu)
            print(f"{(time.time() - t0) / 3600:5.2f} h  GPU{gpu} start  {name}", flush=True)
        time.sleep(2 if args.smoke else 20)
        for gpu, (job, proc, log, started) in list(running.items()):
            if proc.poll() is None:
                continue
            log.close()
            del running[gpu]
            free.append(gpu)
            name, needs, provides, split, _ = job
            status = "done" if proc.returncode == 0 else f"FAILED ({proc.returncode})"
            print(f"{(time.time() - t0) / 3600:5.2f} h  GPU{gpu} {status:11s} {name}  "
                  f"({(time.time() - started) / 60:.1f} min)", flush=True)
            if proc.returncode == 0 and provides:
                picks = tune.select()
                provided.update(p for p in provides if p in picks)
                failed.update(p for p in provides if p not in picks)
            elif proc.returncode != 0:
                failed.update(provides)
        # a job whose inputs can never exist is dropped, so the loop cannot wait forever
        for job in [j for j in pending if any(n in failed for n in j[1])]:
            pending.remove(job)
            print(f"        skipped {job[0]}: missing {[n for n in job[1] if n in failed]}", flush=True)
        if not running and pending and not any(all(n in provided for n in j[1]) for j in pending):
            print(f"nothing runnable; dropping {[j[0] for j in pending]}", flush=True)
            break
        if not running and pending and time.time() > args.deadline:
            print(f"deadline reached; {len(pending)} jobs not started: {[j[0] for j in pending]}", flush=True)
            break
    json.dump(picks, open(os.path.join(test_out, "tuned_v13.json"), "w"), indent=1)
    print(f"pipeline finished in {(time.time() - t0) / 3600:.2f} h; failed: {sorted(failed) or 'none'}", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/multiverse/analyze.py
"""The v13 pre-registered analysis (see PREREGISTRATION.md). Primary tests are paired on seed, two-sided, with
Holm-Bonferroni across all seven; everything else is printed as exploratory.

  python analyze.py <results dir> [more dirs...]   ->  prints tables and writes ANALYSIS.md into the first dir"""
import glob
import json
import math
import os
import sys

import numpy as np
from scipy import stats

PRIMARY = [  # id, task, metric, condition A, condition B (None: one-sample test of A's metric against 0)
    ("P1", "text", "bits_per_char", "single@excitatory", "gru-200"),
    ("P2", "text", "bits_per_char", "single@excitatory", "transformer-64-3"),
    ("P3", "stream", "memory_carry_gain", "single@excitatory:fm", None),
    ("P4", "stream", "gain_delta", "single@excitatory", None),
    ("P5", "text", "bits_per_char", "separate+brain_rigl@excitatory:churn=0.02", "separate@excitatory"),
    ("P6", "neural", "mean_corr", "single", "gru-512"),
    ("P7", "neural", "mean_corr", "single", "gru-128"),
]
EXPLORATORY = {
    "text": ["bits_per_char", "final_slope_per_1k", "novel_8gram", "distinct_4", "judge_bpc", "judge_bpc_real",
             "special_index", "ablate_cv", "ablate_mean", "brain_edge_jaccard", "brain_grown_frac", "params"],
    "stream": ["bpc_windows", "bpc_frozen", "memory_carry_gain", "gain_delta", "gain_dyneval", "gain_hebb_safe",
               "gain_surprise_safe", "gain_confirm_safe", "within_frozen", "within_delta", "params"],
    "neural": ["mean_corr", "median_corr", "top100_corr", "params"],
}


def canonical(condition):
    """Drop the tuned hyperparameters (lr, wd, do) so a condition is named by what it *is*."""
    head, *opts = condition.split(":")
    keep = [o for o in opts if o == "fm" or o.startswith("churn=")]
    return ":".join([head, *keep])


def load(dirs):
    runs = {}
    for d in dirs:
        for f in glob.glob(os.path.join(d, "**", "*.json"), recursive=True):
            data = json.load(open(f))
            if "results" not in data or data.get("args", {}).get("split") != "test":
                continue  # only the reported split counts; tuning runs are validation
            for task, conds in data["results"].items():
                for cond, rs in conds.items():
                    for r in rs:
                        runs.setdefault((task, canonical(cond)), {})[r["seed"]] = r
    return runs


def paired(runs, task, metric, a, b):
    A = runs.get((task, a), {})
    if b is None:
        seeds = sorted(s for s in A if metric in A[s])
        return np.array([A[s][metric] for s in seeds]), seeds
    B = runs.get((task, b), {})
    seeds = sorted(set(A) & set(B))
    return np.array([A[s][metric] - B[s][metric] for s in seeds]), seeds


def holm(ps):
    order = np.argsort(ps)
    adj, running = np.empty(len(ps)), 0.0
    for rank, i in enumerate(order):
        running = max(running, min(1.0, (len(ps) - rank) * ps[i]))
        adj[i] = running
    return adj


def main(dirs):
    runs = load(dirs)
    lines = ["# v13 results", "", "Pre-registered in PREREGISTRATION.md. Test split only. Paired on seed, two-sided,",
             "Holm-Bonferroni across the seven primary tests.", "", "## Primary", "",
             "| id | comparison | metric | n | mean diff | 95% CI | p | p (Holm) | verdict |",
             "|---|---|---|---|---|---|---|---|---|"]
    rows, ps = [], []
    for pid, task, metric, a, b in PRIMARY:
        d, seeds = paired(runs, task, metric, a, b)
        if len(d) >= 2:
            t = stats.ttest_1samp(d, 0.0)
            half = stats.t.ppf(0.975, len(d) - 1) * d.std(ddof=1) / math.sqrt(len(d))
            rows.append((pid, a, b, metric, len(d), d.mean(), half, t.pvalue))
            ps.append(t.pvalue)
        else:
            rows.append((pid, a, b, metric, len(d), float("nan"), float("nan"), float("nan")))
            ps.append(1.0)
    for (pid, a, b, metric, n, mean, half, p), p_adj in zip(rows, holm(np.array(ps))):
        comp = f"`{a}` - `{b}`" if b else f"`{a}` vs 0"
        verdict = "insufficient data" if n < 2 else ("significant" if p_adj < 0.05 else "not significant")
        lines.append(f"| {pid} | {comp} | {metric} | {n} | {mean:+.4f} | ±{half:.4f} | {p:.3g} | {p_adj:.3g} | {verdict} |")
    lines += ["", "Sign: for bits/char a negative difference favours the first condition; for correlation and gains,",
              "a positive one does.", "", "## Exploratory (uncorrected)"]
    for task, metrics in EXPLORATORY.items():
        conds = sorted({c for t, c in runs if t == task})
        if not conds:
            continue
        present = [m for m in metrics if any(m in r for c in conds for r in runs[(task, c)].values())]
        lines += ["", f"### {task}", "", "| condition | n | " + " | ".join(present) + " |",
                  "|---|---|" + "---|" * len(present)]
        for c in conds:
            rs = list(runs[(task, c)].values())
            cells = []
            for m in present:
                v = [r[m] for r in rs if m in r and isinstance(r[m], (int, float))]
                cells.append(f"{np.mean(v):.4g} ± {np.std(v, ddof=1) / math.sqrt(len(v)):.2g}" if len(v) > 1
                             else (f"{v[0]:.4g}" if v else "-"))
            lines.append(f"| `{c}` | {len(rs)} | " + " | ".join(cells) + " |")
    text = "\n".join(lines)
    print(text)
    out = os.path.join(dirs[0], "ANALYSIS.md")
    open(out, "w", encoding="utf-8").write(text + "\n")
    print(f"\nwrote {out}")


if __name__ == "__main__":
    main(sys.argv[1:] or [os.path.join(os.path.dirname(os.path.abspath(__file__)), "results")])

In [ ]:
%%writefile /kaggle/working/multiverse/tests/check_v13.py
"""Correctness checks for the v13 machinery. Each one guards a failure that would corrupt results silently."""
import os
import sys

import numpy as np
import torch
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
import tasks
from connectome import synthetic
from model import FastMemory, GRUBrain, Multiverse, TransformerLM

torch.manual_seed(0)
conn = synthetic(n=120)
V = 11
ok = True


def check(name, cond, detail=""):
    global ok
    ok &= bool(cond)
    print(f"[{'PASS' if cond else 'FAIL'}] {name}{'  ' + detail if detail else ''}")


# 1. the transformer must be causal: changing the last input may not change any earlier prediction
t = TransformerLM(V, V, d_model=32, n_layer=2).eval()
u = F.one_hot(torch.randint(V, (3, 20)), V).float()
u2 = u.clone()
u2[:, -1] = F.one_hot(torch.randint(V, (3,)), V).float()
check("transformer is causal", torch.allclose(t(u)[:, :-1], t(u2)[:, :-1], atol=1e-5))

# 2. streaming the transformer chunk by chunk must equal one pass while the context fits in the block
full = t(u)
a, st = t(u[:, :12], return_state=True)
b = t(u[:, 12:], state=st)
check("transformer streaming == full pass", torch.allclose(torch.cat([a, b], 1), full, atol=1e-5))

# 3. a fresh fast memory is a no-op, so attaching it cannot change what the model starts from
for name, make in [("gru", lambda m: GRUBrain(V, V, 40, memory=m)),
                   ("brain", lambda m: Multiverse(conn, V, V, conn.input_idx, conn.output_idx, 1, "single",
                                                  plastic=True, memory=m))]:
    torch.manual_seed(1)
    plain = make(False).eval()
    torch.manual_seed(1)
    withm = make(True).eval()
    withm.load_state_dict(plain.state_dict(), strict=False)
    check(f"{name}: untrained memory is a no-op", torch.allclose(plain(u), withm(u), atol=1e-6))

# 4. gradient reaches every memory parameter, and state round-trips through return_state
m = GRUBrain(V, V, 40, memory=True)
out, state = m(u, return_state=True)
out.pow(2).mean().backward()
grads = {n: p.grad is not None and p.grad.abs().sum() > 0 for n, p in m.memory.named_parameters()}
check("gradient reaches the memory's output paths", grads["to_out.weight"] and grads["to_inj.weight"])
out2, _ = m(u, state=state, return_state=True)
check("memory state carries across calls", isinstance(state, tuple) and len(state) == 3 and torch.isfinite(out2).all())

# 5. the gated delta rule stores and recalls: write v under k, read back with q = k
mem = FastMemory(8, V, 4, d_mem=16)
S = torch.zeros(1, 16, 16)
k = F.normalize(torch.randn(1, 16), dim=-1)
v = torch.randn(1, 16)
alpha, beta = 1.0, 1.0
S = alpha * (S - beta * torch.bmm(S, k.unsqueeze(-1)) @ k.unsqueeze(1)) + beta * v.unsqueeze(-1) @ k.unsqueeze(1)
check("delta rule recalls what it stored", torch.allclose(torch.bmm(S, k.unsqueeze(-1)).squeeze(-1), v, atol=1e-5))
S2 = S - torch.bmm(S, k.unsqueeze(-1)) @ k.unsqueeze(1) + v.unsqueeze(-1) @ k.unsqueeze(1)
check("rewriting the same pair does not accumulate (Hebbian would double it)", torch.allclose(S2, S, atol=1e-5))

# 6. RigL: gradients exist for absent synapses, growth follows them, brains diverge, invariants hold
torch.manual_seed(0)
mb = Multiverse(conn, V, V, conn.input_idx, conn.output_idx, 3, "separate", plastic=True, rewire="rigl",
                rewire_every=1, rewire_frac=0.05, per_brain_rewire=True)
opt = torch.optim.AdamW(mb.parameters(), 1e-2)
y = torch.randint(V, (3, 20))
loss_fn = lambda o, y: F.cross_entropy(o.reshape(-1, V), y.reshape(-1))
from model import _rigl_scores
g = _rigl_scores(mb, u, y, loss_fn)
check("RigL scores are real gradients", g.shape == (3, 120 * 120) and g.sum() > 0 and mb._probe is None)
for step in range(1, 5):
    loss = loss_fn(mb(u), y)
    opt.zero_grad()
    loss.backward()
    opt.step()
    mb.maybe_rewire(step, 100, u, opt, y=y, loss_fn=loss_fn)
A = mb.alive > 0
d = mb.diagnostics()
signs = torch.sign(mb.w_edge[A.any(0)]) == (1 - 2 * mb.cls[mb.pre[A.any(0)]].float())
check("RigL brains diverge", not bool((A == A[0]).all()), f"jaccard {d['brain_edge_jaccard']:.3f}")
check("E/I signs intact, no live self-loops", bool(signs.all()) and int(((mb.pre == mb.post) & A.any(0)).sum()) == 0)
grown = A & ~mb.is_original[None]
k0 = grown[0].nonzero().squeeze(1)
check("new synapses start at 10% strength", len(k0) > 0 and torch.allclose(
    torch.exp(mb.log_w[0, k0]).mean(), torch.tensor(0.1), atol=0.05))

# 7. evaluation rows align: the per-character bits must match a manual computation
x, yy = tasks._windows(torch.randint(V, (500,)), 20)
model = GRUBrain(V, V, 30).eval()
bits = tasks._eval_nll(model, x, yy, V, bs=7)
with torch.no_grad():
    manual = F.cross_entropy(model(F.one_hot(x, V).float()).reshape(-1, V), yy.reshape(-1), reduction="none")
check("deterministic eval aligns rows", torch.allclose(bits, manual / np.log(2), atol=1e-5))

# 8. the n-gram judge prefers real text to shuffled text
text = ("the quick brown fox jumps over the lazy dog and the cat sat on the mat " * 60)
judge = tasks._CharJudge(text[:3000])
real, shuffled = text[3000:3400], "".join(np.random.default_rng(0).permutation(list(text[3000:3400])))
check("judge scores real text better than shuffled", judge.bpc(real) < judge.bpc(shuffled),
      f"{judge.bpc(real):.2f} vs {judge.bpc(shuffled):.2f} bits")

print("\nALL PASS" if ok else "\nFAILURES ABOVE")
sys.exit(0 if ok else 1)

In [ ]:
%%writefile /kaggle/working/multiverse/PREREGISTRATION.md
# v13 pre-registration

Committed before the reported run. Hyperparameters are chosen on the **validation** split by `tune.py` (picks in
`tuned_v13.json`); every reported number comes from the **test** split, which nothing is tuned on. The notebook
records the git commit it was built from.

## Why v13 exists

A self-critique of v1–v12 found: untuned baselines, 3 seeds with no multiple-comparison correction, random-batch
evaluation, bolt-on fast weights that were never trained, a "gradient-free" rule that is algebraically a last-layer
gradient step, a growth rule described wrongly, specialisation measured on a broken model with misaligned rows,
and tuning on a synthetic stand-in. v13 fixes each of these and asks four narrow questions.

## Design

- **Data.** tinyshakespeare: train 0–80%, val 80–90%, test 90–100% (the test region equals the old v1–v12 region).
  MICrONS functional window: train 0–70%, val 70–80%, test 80–100% (the region v6 scored). Neuron selection and
  stimulus normalisation use the training part only.
- **Evaluation.** Deterministic: every character of the split in fixed 64-character windows (text); every
  40-frame window (neural). The same characters for every model.
- **Training.** One recipe for every model: AdamW, 100-step warmup, cosine decay to 10%, gradient clip 1.0,
  weight decay on weight matrices only. 4,000 steps (text, stream), 2,000 (neural). `sample_prob = 0`.
- **Tuning.** `tune.py`, validation split, seed 0: learning rate × weight decay for every family, plus dropout for
  the GRUs and transformers (the brain does not overfit: v12 final slope −0.013). Q3's six-brain models and every
  `:fm` model reuse the tuned values of their single-brain / core counterpart; they are too slow to tune on CPU.
  The stream task's delta-rule rate and dynamic-evaluation learning rate are tuned the same way.
- **Seeds.** 0–4 for every condition. Comparisons are paired on seed.
- **Run order.** Primary conditions run first on each GPU, so a session that nears Kaggle's 12-hour limit (no new
  run starts after 11.25 h) can only lose exploratory runs.
- **Tests.** Two-sided paired t-test on per-seed differences (one-sample t-test for within-model gains),
  α = 0.05, **Holm–Bonferroni across the seven primary tests**. Reported with mean difference, 95% CI, raw and
  adjusted p. Anything not listed as primary is exploratory and is labelled as such.

## Parameter matching

| task | brain | matched baseline | larger baseline |
|---|---|---|---|
| text | `single@excitatory` 170k | `gru-200` 173k, `transformer-64-3` 163k | `gru-512` 923k, `transformer-128-4` 818k |
| neural | `single` (L5 readout, as v6) 404k | `gru-128` 400k | `gru-512` 2.19M (v6's comparator) |

## Primary hypotheses

| id | question | comparison | metric |
|---|---|---|---|
| P1 | Q1 baselines | `single@excitatory` vs `gru-200` | test bits/char |
| P2 | Q1 baselines | `single@excitatory` vs `transformer-64-3` | test bits/char |
| P3 | Q2 learning on the job | `single@excitatory:fm`: memory carried across the stream vs wiped every chunk | `memory_carry_gain` > 0 |
| P4 | Q2 learning on the job | `single@excitatory`: readout delta rule vs frozen (replicates v12) | `gain_delta` > 0 |
| P5 | Q3 differentiation | `separate+brain_rigl@excitatory:churn=0.02` vs `separate@excitatory` | test bits/char |
| P6 | Q4 neural | `single` vs tuned `gru-512` (tests v6's claim as stated) | test mean correlation |
| P7 | Q4 neural | `single` vs tuned `gru-128` (parameter-matched) | test mean correlation |

P3 is the clean test of learning from a stream without backprop: the fast memory's writes are delta-rule updates
inside the forward pass, trained end to end (Gated DeltaNet recurrence). Pretraining is stateful so a memory could
learn to persist. `memreset` wipes the memory at every 32-character chunk while keeping the network's own state,
so the difference is what the memory carries forward.

## Exploratory (reported, uncorrected)

- Q1: `gru-512`, `transformer-128-4`; final learning-curve slopes (overfitting check).
- Q2: `gain_dyneval` (full backprop on the stream, the ceiling) and the share of it the delta rule reaches — the
  delta rule is one SGD step on the readout, so this measures how far a *local* step gets, not a new principle;
  `memory_carry_gain` for `gru-200:fm`; clamped Hebbian controls (`hebb_safe`, `surprise_safe`, `confirm_safe`),
  including whether surprise or confirm is less harmful (v12 reversed the synthetic result); within-stream slopes.
- Q3: 10% churn; specialisation on the frozen model (`special_index`, `ablate_cv`) for clones vs differentiated;
  structural divergence (`brain_edge_jaccard`, `brain_grown_frac`).
- Creativity: `novel_8gram` and `distinct_4` read beside `judge_bpc` (a Witten-Bell character 5-gram model of the
  training text) relative to `judge_bpc_real` (the same judge on real held-out text). Novelty only counts at
  comparable judged quality.

## Decision rules

- P5 not significant **and** `special_index` of differentiated brains no higher than clones → the differentiation
  line is closed and no mixture-of-experts router is built.
- P3 not significant → a trained fast memory does not produce stream learning in this setup; reported as a null.
- P6/P7 not significant → the v6 claim that the sparse E/I architecture predicts cortex better than a GRU is
  withdrawn.

## Out of scope, and why

- **Whether the measured wiring matters.** The test that could show it — model neuron *i* must predict recorded
  neuron *i* — needs the MICrONS functional coregistration tables. They are served through CAVE, which we cannot
  access, and neither our connectome file (`connectivity/*` only) nor our recording window contains them. v13
  makes no claim either way about the connectome itself.
- **Mixture-of-experts router.** Conditional on Q3, per the decision rule above.
- **A larger corpus.** v13 changes methodology, not data, so results stay comparable with v1–v12.

## Amendment (before any test-split result existed)

Tuning moved from the local CPU into the Kaggle session itself (`pipeline.py`), to start the run sooner. Nothing
else changed: the same grids, validation split and seed 0; `tune.select` picks the best validation score per
family, and the test-split conditions are built from those picks inside the session. The pipeline starts each
question as soon as its picks exist and splits reported runs per seed so the two GPUs balance. No test-split number
was seen before this amendment; the eight `gru-200` local tuning runs are discarded, not reused.

In [ ]:
import glob, shutil
# data comes only from the attached private dataset: no runtime download can fail the run
find = lambda name: glob.glob(f"/kaggle/input/**/{name}", recursive=True)[0]
shutil.copy(find("microns_v1181_axon.npz"), "/kaggle/working/multiverse/data/microns_v1181_axon.npz")
shutil.copy(find("tinyshakespeare.txt"), "/kaggle/working/char_lm/tinyshakespeare.txt")
NEURAL = find("functional_scan.npz")
print("neural data:", NEURAL, os.path.getsize(NEURAL) // 2**20, "MB")

In [ ]:
import subprocess, sys
# the same 14 correctness checks that passed locally, re-run in this environment before any GPU hour is spent
r = subprocess.run([sys.executable, "/kaggle/working/multiverse/tests/check_v13.py"], capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-2000:])
assert r.returncode == 0, "correctness checks failed - not launching"

In [ ]:
import subprocess, sys, time
# tuning (validation split) then the pre-registered test runs, scheduled across both GPUs by pipeline.py
log = open("/kaggle/working/pipeline.log", "w")
proc = subprocess.Popen([sys.executable, "-u", "/kaggle/working/multiverse/pipeline.py", "--deadline", str(DEADLINE), "--neural", NEURAL,
                         "--gpus", str(max(torch.cuda.device_count(), 1)), "--seeds", str(SEEDS),
                         "--out", "/kaggle/working/multiverse/results"], stdout=log, stderr=subprocess.STDOUT)
seen = 0
while proc.poll() is None:
    time.sleep(300)
    lines = open("/kaggle/working/pipeline.log").read().splitlines()
    print("\n".join(l for l in lines[seen:] if " h  GPU" in l or "skipped" in l or "deadline" in l), flush=True)
    seen = len(lines)
print(open("/kaggle/working/pipeline.log").read()[-4000:])

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "/kaggle/working/multiverse/analyze.py", "/kaggle/working/multiverse/results/v13"], capture_output=True, text=True,
                   env={**os.environ, "PYTHONIOENCODING": "utf-8"})
print(r.stdout, r.stderr[-2000:])

The pre-registered verdicts are in `multiverse/results/v13/ANALYSIS.md`; picks in `results/v13/tuned_v13.json`.